# Financial Trading with Python, 2nd Edition
Cordell L. Tanny, CFA, FRM, FDP

## Chapter 2: Setting up a Python Quantitative Workflow
### Notebook 2.1: Pandas and Numpy for Finance
Version: 2.0

Date of last revision: December 10, 2025


Suggested additional resource: https://www.w3schools.com/python

**Pandas and DataFrames**

*Note and recommendation: Very often, you might want to experiment on your own as you go through this notebook. We recommend you save a copy of this notebook before you start adding cells or changing anything. This way you always have a pristine copy to go back to.*

In [1]:
import os
import pandas as pd

# ---- Data source settings ----
# False: use the data saved with the book, so your results match the printed ones.
# True:  download fresh data from Yahoo! Finance (your numbers will differ from the book).
USE_LIVE_DATA = False
REPO_RAW = ("https://raw.githubusercontent.com/PacktPublishing/"
            "Hands-on-Financial-Trading-with-Python-Second-Edition/main/FINAL")
DATA_FOLDER = "chapter02"   # repo folder that holds this notebook's saved files

def load_saved_data(file_name: str) -> pd.DataFrame:
    """
    Load a data file saved with the book.

    1. Use the file if it is already in Colab (for example, you uploaded it).
    2. Otherwise, download it from the book's GitHub repository.
    """
    if not os.path.exists(file_name):
        !wget -q {REPO_RAW}/{DATA_FOLDER}/{file_name}
    if not os.path.exists(file_name):
        raise FileNotFoundError(f"{file_name} is not in Colab or the repo. Set USE_LIVE_DATA = True.")
    return pd.read_parquet(file_name)

## 1.0 Introduction

### 1.1 What is a DataFrame?

A DataFrame is a two-dimensional, tabular data structure in Pandas, similar to an Excel spreadsheet or SQL table. It is the most commonly used object in Pandas and is highly flexible for analyzing and manipulating data.

For financial time series, a DataFrame is often used to organize date-based data such as stock prices, returns, or other financial indicators.


---

**Key Parts of a DataFrame**

A Pandas DataFrame is made up of the following key components:

1. Rows

- Represent individual records or observations.
Indexed by a unique identifier (default is an integer index, but for financial time series, this is typically a DatetimeIndex).

2. Columns

- Represent the different features or variables of the dataset (e.g., stock prices, volumes, returns).
Each column has a name (header) and a specific data type (e.g., float, int, datetime).

3. Index

- A special label for rows that allows quick access and alignment of data.
For financial time series, the index is often a datetime object representing dates and times.
4. Data

- The actual values stored in the DataFrame, organized in rows and columns.
Can contain numeric data (prices, returns), strings (tickers, sectors), or dates (timestamps).



Let's start by creating a simple dataframe ("df") and looking at its structure and elements.

Do not worry too much about creating a df with sample data right now as will come back to that later.
You can examine the documentation for pd.DataFrame() [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html). But this is something you should absolutely know how to do because creating a dataframe from a dictionary will occur very often.

Our first task is to import pandas and get used to the most common parts of a df and common methods.

In [2]:
# Import pandas as pd (pd is the accepted convention)
import pandas as pd

# We create a sample dataframe by converting a dictionary to a dataframe
# with the built in pd.DataFrame()
data = {
    'spy': [150.25, 151.30, 152.10, 149.80, 148.90],
    'MSFT': [299.50, 300.00, 302.20, 298.20, 297.00],
    'GOOGL': [2800.25, 2825.00, 2830.50, 2790.50, 2780.30]
}

# This creates a list of datetime objects.
dates = pd.date_range(start='2023-01-01', periods=5)

# Create DataFrame
df = pd.DataFrame(data, index=dates)

print(df)

               spy   MSFT    GOOGL
2023-01-01  150.25  299.5  2800.25
2023-01-02  151.30  300.0  2825.00
2023-01-03  152.10  302.2  2830.50
2023-01-04  149.80  298.2  2790.50
2023-01-05  148.90  297.0  2780.30


We can see that it is a table. But let's breakdown the elements so that you are comfortable with the nomenclature.

1. Rows
- Each row represents the stock prices for a specific date.
- The index (e.g., 2023-01-01) uniquely identifies each row.
2. Columns
- SPY, MSFT, and GOOGL are the column labels (representing stock tickers).
- Each column contains the prices for the corresponding stock.
3. Index
- The index is a DatetimeIndex (2023-01-01, 2023-01-02, etc.).
- It provides an efficient way to access rows by date.
4. Data
- The numbers (e.g., 150.25, 299.50) are the actual stock prices for each ticker and date.

### 1.2 What is a DateTime Object?
A datetime object in Python represents a specific date and time, including components like year, month, day, hour, minute, second, and microsecond. Python's datetime module provides tools for working with dates and times, making it easy to handle, manipulate, and format temporal data.

Here’s an example of creating and printing a datetime object:


In [3]:
from datetime import datetime

# Create a datetime object
dt = datetime(2023, 1, 1, 12, 30, 45)
print(dt)
# Output: 2023-01-01 12:30:45


2023-01-01 12:30:45


**Why Are Datetime Objects Important?**

1. Time-Based Analysis:

- Many datasets (especially financial ones) are indexed or recorded by time. Understanding how to work with time data is essential for analyzing trends and patterns over specific periods.

2. Accuracy in Financial Data:

- Investments rely on precise timestamps for trades, prices, and economic data. Datetime objects allow exact tracking of when events occur.

3. Efficient Filtering and Slicing:

- You can easily filter or slice data based on specific date ranges using datetime objects. For example, extracting all stock prices for a particular year or month.

4. Time Zones:

- Financial markets operate in different time zones. Datetime objects can handle timezone-aware calculations, ensuring consistency across regions.

In [4]:
# Sample DataFrame of stock prices
data = {'spy': [150, 151, 152, 153, 154],
        'MSFT': [299, 300, 301, 302, 303]}
dates = pd.date_range('2023-01-01', periods=5)

df = pd.DataFrame(data, index=dates)

# Filter stock prices for January 3, 2023
print(df.loc['2023-01-03'])  # Note that we don't need to specify the time and minutes!



spy     152
MSFT    301
Name: 2023-01-03 00:00:00, dtype: int64


**Creating a Datetime Object with GMT Timezone**

To add timezone information, you can use the pytz library or datetime.timezone. Here’s how to create a datetime object in GMT:

In [5]:
from datetime import datetime, timezone, timedelta

# Create a timezone-aware datetime object in GMT
gmt_time = datetime(2023, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
print(gmt_time)


2023-01-01 12:00:00+00:00


So, why are we showing you this?

Very often, when you download time series data from different vendors, you might see time zone information in the datetime index. This can cause a lot of problems in your code.

We want you to get in the habit of examining all important information when you download time series data from day 1!

**How to remove the timezone information**

To remove the timezone information, convert the timezone-aware datetime object to a naive datetime object using the .replace() method:

In [6]:
# Remove timezone information
naive_time = gmt_time.replace(tzinfo=None)
print(naive_time)


2023-01-01 12:00:00


Now, the +00:00 has been removed, and the datetime object is timezone-naive.

**Key Differences: Timezone-Aware vs. Naive**
1. Timezone-Aware:

- Contains timezone information (e.g., +00:00 for GMT).
- Useful for working with data across multiple time zones.
- Example: 2023-01-01 12:00:00+00:00

2. Timezone-Naive:

- Does not include timezone information.
- Easier for basic analysis but risky when working with data from different time zones.
- Example: 2023-01-01 12:00:00



---




### 1.3 Key Attributes of a DataFrame

#### 1.3.1 `.index`: Access the index (e.g., dates for financial time series).

The `.index` attribute of a Pandas DataFrame provides the labels for the rows. In financial time series data, it is typically a DatetimeIndex, which is crucial for analyzing and manipulating data based on time.

In [7]:
print(df.index)

DatetimeIndex(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04',
               '2023-01-05'],
              dtype='datetime64[ns]', freq='D')


So what do we see in the output and what should you look out for?

1. DatetimeIndex:

- Indicates that the index contains datetime objects.
- Essential for time-based analysis.

2. Values ('2023-01-01', '2023-01-02', ...):

- These are the individual labels for each row, typically representing dates.
dtype='datetime64[ns]':

3. Specifies the data type as 64-bit datetime objects.

4. freq='D':

- Indicates the frequency of the index. In this case, D means daily.
- Frequencies are critical for understanding gaps or irregularities in the data.

Note that this is specific to a datetime index. We will explore what the index will look like for other types of tabular data later. Right now, we want you to get comfortable with timeseries data!

**Why Do We Check `.index`?**
1. To Understand the Structure of the Data:

- The .index tells you what kind of labels are used for rows (e.g., dates, integers).
- For time series, a DatetimeIndex confirms that the data is time-based.

2. For Consistency in Operations:

- Ensures the index aligns with your expectations (e.g., date frequency).
- Misaligned indices can cause errors when merging or slicing data.

3. To Enable Date-Based Operations:

- A DatetimeIndex allows powerful slicing, filtering, and resampling of data based on dates.

So, what should you look for when checking `.index`?

1. Is the index in the correct format (e.g., datetime, strings, etc.)
2. Is the frequency correct (for timeseries)
3. Are the dates in order? -> More on sorting later
4. Are there duplicates?

#### 1.3.2 The `.columns` attribute

The .columns attribute of a Pandas DataFrame provides the labels for the columns, representing the variables or features in the dataset.




In [8]:
data = {'spy': [150, 151, 152, 153, 154],
        'MSFT': [299, 300, 301, 302, 303]}
dates = pd.date_range('2023-01-01', periods=5)

df = pd.DataFrame(data, index=dates)

print(df.columns)

Index(['spy', 'MSFT'], dtype='object')


**Why Do We Check .columns?**

1. To Understand the Features:

- Identifies what data is stored in each column (e.g., stock tickers, returns, prices).
2. For Data Validation:

- Confirms that all expected variables are present and correctly labeled.
- Helps avoid errors caused by typos or missing columns.

3. For Efficient Data Manipulation:

- Enables dynamic operations, like renaming columns, selecting specific ones, or filtering.

Let's examine the output from the previous cell:

**Key Components:**

1. Column Labels:

- ['spy', 'MSFT', 'GOOGL'] are the column names, often representing stock tickers, metrics, or variables.

2. Data Type (dtype='object'):

- Indicates the labels are strings (object type in Pandas).

**When to use `.columns`:**
1. Inspect the Column Names:
- Quickly check what features are available in the dataset.
2. Select specific columns:
- Use column names to slice the DataFrame
- Example: `print(df[['spy', 'MSFT]])`
3. Renaming columns
- Rename columns for clarity or standardization:
4. Check for Missing or Extra Columns:
- Validate the presence of required columns:

**What to look for in the output:**

1. Correct labels
2. Unique labels

#### 1.3.3 The `.values` attribute
The .values attribute provides the underlying data of the DataFrame as a NumPy array. It contains the actual numbers, strings, or other types stored in the DataFrame.


In [9]:
print(df.values)

[[150 299]
 [151 300]
 [152 301]
 [153 302]
 [154 303]]


In [10]:
print(type(df.values))

<class 'numpy.ndarray'>


Uh-oh! It's an array! We haven't seen those yet!

Don't worry about the fact that the output is an array. However it is very important, since many machine learning algorithms will only accept arrays as inputs, not dataframes.

For now, this is what you should know:

1. Produces an array as an ouput.
2. Allows you to quickly inspect the data from a df without column and index labels.
3. It can also be used to select a subset of data from the dataframe (more on this later)

#### 1.3.4 The `.shape` attribute

The .shape attribute provides the dimensions of the DataFrame as a tuple (number of rows, number of columns).


In [11]:
print(df)
print(df.shape)

            spy  MSFT
2023-01-01  150   299
2023-01-02  151   300
2023-01-03  152   301
2023-01-04  153   302
2023-01-05  154   303
(5, 2)


In [12]:
# But also note that we can use .len to get the number of rows
print(len(df))

5


**Why Do We Check .shape?**

1. To Understand the Dataset Size:
- Confirms how many rows (observations) and columns (features) the dataset contains.
2. For Validation:
- Ensures the dataset matches expected dimensions before processing.
3. To Debug Issues:
- Quickly spot missing or extra rows/columns.

**When to use `.shape`**
1. Check dataset size.
2. Validate after filtering:
- Check the size after applying filters or trasnformations.
3. Handle large datasets:
- Verify dimensions before running memory-intensive operations.


#### 1.3.5 Summary of key attributes

These key attributes provide crucial insights into the structure and content of a DataFrame:

- `.index`: Validates the time-based index for financial data.
- `.columns`: Confirms the presence and labeling of variables.
- `.values`: Accesses raw data for advanced operations.
- `.shape`: Ensures the dataset has the correct dimensions.

By checking these attributes, you can efficiently validate, manipulate, and analyze financial datasets. Let me know if you'd like additional examples or extensions!



---



### 1.4 What is an Attribute? What is a Method?

#### 1.4.1 What Is an Attribute?
An attribute is a property or characteristic of an object. It provides information about the object and does not require parentheses to access.

Think of it as a descriptor or label for the object.
Accessed directly using the dot notation: object.attribute.

We've already seen examples of attributes in the previous section.

#### 1.4.2 What Is a Method?

A method is a function associated with an object. It performs an action or operation on the object, often returning a result or modifying the object.

Think of it as an action or behavior the object can perform.
Methods require parentheses, even if they don't take arguments: object.method().

We will turn to methods now, and show you the most important ones you need to get comfortable with right away.


---



### 1.5 Key DataFrame Methods

We will go through this section by examining a real financial time series.
We will download historical pricing information for Apple for a 20 year period.
This way, as we work through each method, you will gain the practical experience of what to expect.  

#### 1.5.1 Downloading Data from Yahoo! Finance Using yFinance

[yfinance](https://pypi.org/project/yfinance/) is a Python package specifically created to easilly retrieve information from Yahoo! Finance.
And guess what data type the output will be?
That's right! A dataframe.

Google Colab is equipped with yfinance, so there is no need to install it.

In [13]:
# import yfinance
import yfinance as yf  # convention

# It is good practice to specify the ticker, start date and end date as variables
ticker = 'SPY'
start_date = '2005-01-01'  # Notice that we are setting the date as a string and not a datetime object
end_date = '2024-11-30'

# retrieve the prices
if USE_LIVE_DATA:
    df_spy = yf.download(tickers=ticker, start=start_date, end=end_date, auto_adjust=False)
else:
    df_spy = load_saved_data('ch02_nb01_spy_2005_2024.parquet')

We are going to start by looking at the attributes that we just covered.

In [14]:
# Examine the index
df_spy.index

DatetimeIndex(['2005-01-03', '2005-01-04', '2005-01-05', '2005-01-06',
               '2005-01-07', '2005-01-10', '2005-01-11', '2005-01-12',
               '2005-01-13', '2005-01-14',
               ...
               '2024-11-15', '2024-11-18', '2024-11-19', '2024-11-20',
               '2024-11-21', '2024-11-22', '2024-11-25', '2024-11-26',
               '2024-11-27', '2024-11-29'],
              dtype='datetime64[ns]', name='Date', length=5012, freq=None)

We can see:
1. It is in datetime (eventhough we put the dates in as strings)
2. The index name is 'Date'
3. We have 5012 rows
4. The frequency wasn't determined (we will address this later).

Let's confirm the length:

In [15]:
print(len(df_spy))

5012


In [16]:
# Now let's take a look at the shape
print(df_spy.shape)

(5012, 6)


Notice that we have 5 columns. Let's find out what they are using the `.columns` attribute.

In [17]:
print(df_spy.columns)

MultiIndex([('Adj Close', 'SPY'),
            (    'Close', 'SPY'),
            (     'High', 'SPY'),
            (      'Low', 'SPY'),
            (     'Open', 'SPY'),
            (   'Volume', 'SPY')],
           names=['Price', 'Ticker'])


MultiIndex....What is that???

We aren't going to get into MultiIndex just yet. But thnk of it as having two levels of column headings, one level is the ticker and the other is something related to the pricing information for each day.

We will drop the ticker level to make our life easier.
Don't worry about this code yet; just know that it is a result of the way Yahoo! Finance returns the data.

In [18]:
# Remove the second level of the MultiIndex
df_spy.columns = df_spy.columns.get_level_values(0)
df_spy.columns

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

#### 1.5.2 Investigating the data

We will start with these four important methods, and look at what they are, and when you should use. These are typically the first things you would do to investigate a dataframe.

1. `.info()`
2. `.describe()`
3. `.head()`
4. `.tail()`


---



##### 1. `.info()`

**What It Does:**
- Provides a concise summary of the DataFrame, including:
 - Index type and range.
 - Number of non-null values in each column.
 - Data types of each column.
 - Memory usage of the DataFrame.

**Why It’s Useful:**
- Helps you understand the structure of the dataset.
- Quickly identifies missing values and data types.
- Useful for large datasets to check memory efficiency.

In [19]:
df_spy.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5012 entries, 2005-01-03 to 2024-11-29
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  5012 non-null   float64
 1   Close      5012 non-null   float64
 2   High       5012 non-null   float64
 3   Low        5012 non-null   float64
 4   Open       5012 non-null   float64
 5   Volume     5012 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 274.1 KB


We can see each of the column names as well as their data types. It will also tell us if there are missing values, based on the non-null count. It also provides the date range in the index.
There is a lot of useful information here!

**Key Information to Look For:**
- Non-Null Count: Indicates whether there are missing values.
- Dtypes: Ensures columns have the correct data type (e.g., float64 for prices).
- Memory Usage: Helps optimize memory when working with large datasets.


---



##### 2. `.describe()`
**What It Does:**
- Provides summary statistics for numeric columns in the DataFrame, such as:
 - count, mean, std, min, 25%, 50%, 75%, max.

**Why It’s Useful:**

- Gives a quick snapshot of the distribution of numeric data.
- Identifies outliers or anomalies in the data.
- Useful for assessing the scale and variability of financial metrics.


In [20]:
df_spy.describe()

Price,Adj Close,Close,High,Low,Open,Volume
count,5012.000000,5012.000000,5012.000000,5012.000000,5012.000000,5.012000e+03
mean,202.976089,233.659220,234.932807,232.210064,233.631069,1.259418e+08
std,130.033205,125.240091,125.812421,124.551928,125.211641,9.140181e+07
min,49.557522,68.110001,70.000000,67.099998,67.949997,1.527000e+07
25%,96.629438,131.360001,132.010002,130.550003,131.202499,6.613088e+07
50%,162.964241,198.059998,198.915001,197.235001,198.024994,9.548135e+07
75%,269.914665,298.414993,299.862495,297.032501,298.525002,1.548838e+08
max,588.958374,602.549988,603.349976,599.380005,600.460022,8.710263e+08


As you can see, this method provides descriptive statistics for all numeric data columns.

**Key Information to Look For:**
- Mean and Median (50%): Shows the central tendency of the data.
- Min and Max: Helps spot potential outliers.
- Standard Deviation (std): Indicates the variability of prices.

##### 3. `.head()`
**What It Does:**
- Displays the first few rows of the DataFrame (default is 5 rows).

**Why It’s Useful:**
- Provides a quick preview of the dataset.
- Useful for verifying data imports and checking initial rows.

Note: By default, it will show you the first 5 rows. You can specify how many rows to return inside the parentheses:

df_spy.head(15) would return the first 15 rows

**Key Information to Look For:**
- Correctness of Columns: Ensure columns and their values are aligned.
- Initial Data: Confirm that dates and financial metrics look valid.



---



In [21]:
df_spy.head()

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2005-01-03,80.973534,120.300003,121.760002,119.900002,121.559998,55748000
2005-01-04,79.984070,118.830002,120.540001,118.440002,120.459999,69167600
2005-01-05,79.432137,118.010002,119.250000,118.000000,118.739998,65667300
2005-01-06,79.836021,118.610001,119.150002,118.260002,118.440002,47814700
2005-01-07,79.721573,118.440002,119.230003,118.129997,118.970001,55847700


In [22]:
df_spy.head(15)

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2005-01-03,80.973534,120.300003,121.760002,119.900002,121.559998,55748000
2005-01-04,79.984070,118.830002,120.540001,118.440002,120.459999,69167600
2005-01-05,79.432137,118.010002,119.250000,118.000000,118.739998,65667300
2005-01-06,79.836021,118.610001,119.150002,118.260002,118.440002,47814700
2005-01-07,79.721573,118.440002,119.230003,118.129997,118.970001,55847700
2005-01-10,80.098518,119.000000,119.459999,118.339996,118.339996,56563300
2005-01-11,79.546547,118.180000,118.739998,117.989998,118.639999,63099700
2005-01-12,79.809067,118.570000,118.839996,117.519997,118.400002,72720500
2005-01-13,79.169670,117.620003,118.730003,117.500000,118.639999,55537500




---


##### 4. `.tail()`
**What It Does:**
- Displays the last few rows of the DataFrame (default is 5 rows).
**Why It’s Useful:**
- Useful for examining recent data, especially in time series.
- Helps confirm the end of the dataset is as expected after filtering or transformations.

This behaves in the exact same way as `.head()`

In [23]:
df_spy.tail(10)

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2024-11-15,572.537354,585.750000,590.200012,583.859985,589.719971,75988800
2024-11-18,574.883240,588.150024,589.489990,585.340027,586.219971,37001700
2024-11-19,576.984619,590.299988,591.039978,584.030029,584.710022,49412000
2024-11-20,577.180237,590.500000,590.789978,584.630005,590.380005,50032600
2024-11-21,580.278625,593.669983,595.119995,587.450012,593.400024,46750300
2024-11-22,582.077271,595.510010,596.150024,593.150024,593.659973,38226400
2024-11-25,584.051575,597.530029,600.859985,595.200012,599.520020,42441400
2024-11-26,587.101318,600.650024,601.330017,598.070007,598.799988,45621300
2024-11-27,585.322327,598.830017,600.849976,597.280029,600.460022,34000200



---
#### 1.5.3 Mean, Standard Deviation and Percent Change

These methods allow you to perform calculations on a single column or subset of columns, helping analyze key statistics and trends in financial time series data.

##### `.mean()`

- What It Does: Calculates the mean (average) value for numeric columns.
- Why It’s Useful: Provides the central tendency of a specific column or the entire dataset. And quickly!

In [24]:
# Find the average values of every column.
# We will store it in a variable.
averages = df_spy.mean()
print(averages)

Price
Adj Close    2.029761e+02
Close        2.336592e+02
High         2.349328e+02
Low          2.322101e+02
Open         2.336311e+02
Volume       1.259418e+08
dtype: float64


Please take a minute to look at the results here. Is the format of the output what you expected?
We can see that it is a data type float, which is good, and we can see the average values for each column.

But let's do a quick check, since this will introduce a key concept that you must learn when workig with pandas.

In [25]:
print(type(averages))

<class 'pandas.core.series.Series'>


It's a series????
What is a series?

##### Difference between a dataframe and a series
1. Definition
- Series: A one-dimensional labeled array in Pandas. It is similar to a list or NumPy array but with labels (index) for each value.
- DataFrame: A two-dimensional labeled data structure in Pandas, similar to an Excel spreadsheet or SQL table, consisting of rows and columns.

2. Structure
- Series:
 - Contains a single column of data.
 - Has one axis (index).
 - Can hold any data type (e.g., numeric, strings, datetime).
- DataFrame:
 - Contains multiple rows and columns.
 - Has two axes:
 - Rows: Indexed by default or custom labels.
 - Columns: Labeled for each variable.

 Always remember that a dataframe will have shape (n, m) and a series will have shape (n,).

 This is really important to remember because not all dataframe methods will work on a series and vice-versa. So, you must always pay attention to the format of your output. Normally, you will want your output to be a dataframe.

 Let's take a closer look at the output:

In [26]:
averages

,0
Price,
Adj Close,2.029761e+02
Close,2.336592e+02
High,2.349328e+02
Low,2.322101e+02
Open,2.336311e+02
Volume,1.259418e+08


Notice that there is no column heading. It just shows a zero.
We can name the column with:

`averages.name = 'spy Stats'`

In [27]:
averages.name = 'spy Stats'
averages

,spy Stats
Price,
Adj Close,2.029761e+02
Close,2.336592e+02
High,2.349328e+02
Low,2.322101e+02
Open,2.336311e+02
Volume,1.259418e+08


We will come back to working with series another time.
Let's continue with important dataframe methods.

Just remember to pay attention to the data type of the output.

Notice that when you called .mean(), it computed the averages across the rows. Meaning, it applied it a column, and took the average of all rows in that column. This is the same thing as saying "column-wise".

What if you wanted to calculate the mean "row-wise", or using all values in one row for all columns?

Then you would have to do this:

`df_spy.mean(axis=1)`





##### The axis parameter

The axis parameter in Pandas controls whether operations like .mean(), .sum(), .std(), and others are applied row-wise or column-wise.

**What axis=0 and axis=1 Mean**

1. axis=0 (Default):

- The operation is applied column-wise.
- For each column, Pandas computes the result across all rows.
- Think: "Down the columns."

2. axis=1:

- The operation is applied row-wise.
- For each row, Pandas computes the result across all columns.
- Think: "Across the rows."


In [28]:
averages_axis_1 = df_spy.mean(axis=1)
averages_axis_1

,0
Date,
2005-01-03,9.291427e+06
2005-01-04,1.152803e+07
2005-01-05,1.094464e+07
2005-01-06,7.969209e+06
2005-01-07,9.308042e+06
...,...
2024-11-22,6.371560e+06
2024-11-25,7.074063e+06
2024-11-26,7.604048e+06


Look at how different the output is!
We are getting a time series that shows the average of all columns for each date!

This will be useful down the road. For now, we will be dealing with axis=0, which is the default.

**Note that this axis parameter works for most pandas methods.**

---
##### `.std()`

- What It Does: Calculates the standard deviation, a measure of variability in the data.
- Why It’s Useful: Indicates how much prices fluctuate around the mean.

Pandas is so powerful because it handles the most common transformations and calculations for us.
Standard deviation is just one of many examples.

In [29]:
std = df_spy.std()
print(std)

Price
Adj Close    1.300332e+02
Close        1.252401e+02
High         1.258124e+02
Low          1.245519e+02
Open         1.252116e+02
Volume       9.140181e+07
dtype: float64


We know that the scientific notation seen in the output isn't ideal, but let's leave it for now.

---

##### `.pct_change()`
- What It Does: Calculates the percentage change between consecutive rows.
- Why It’s Useful: Converts raw prices into returns, which are crucial for financial analysis.

The percent change method is one of the most useful methods for dataframes and you will be using it quite often.
The beautiful thing about it, is that you can specify the period over which to measure the percent change within the brackets!
By default, it will calculate the percent change of all columns in relation to the previous row.

If, for example, you wanted to measure the percent change vs. 3 periods ago, you would do:

`.pct_change(3)`

In [30]:
df_pct_change = df_spy.pct_change()

print(df_pct_change)

Price       Adj Close     Close      High       Low      Open    Volume
Date                                                                   
2005-01-03        NaN       NaN       NaN       NaN       NaN       NaN
2005-01-04  -0.012220 -0.012219 -0.010020 -0.012177 -0.009049  0.240719
2005-01-05  -0.006901 -0.006901 -0.010702 -0.003715 -0.014279 -0.050606
2005-01-06   0.005085  0.005084 -0.000839  0.002203 -0.002526 -0.271864
2005-01-07  -0.001434 -0.001433  0.000671 -0.001099  0.004475  0.168003
...               ...       ...       ...       ...       ...       ...
2024-11-22   0.003100  0.003099  0.001731  0.009703  0.000438 -0.182328
2024-11-25   0.003392  0.003392  0.007901  0.003456  0.009871  0.110264
2024-11-26   0.005222  0.005221  0.000782  0.004822 -0.001201  0.074924
2024-11-27  -0.003030 -0.003030 -0.000798 -0.001321  0.002772 -0.254730
2024-11-29   0.006212  0.006212  0.004161  0.003516 -0.001332 -0.112435

[5012 rows x 6 columns]


You can see that each value now shows the percent change compared to the previous row, and in decimal format (1% = 0.01)

You should also notice a couple of things from the output:
1. That when you try to output an entire dataframe, it will show you the first 5 rows, then ... and then the last 5 rows.
2. The first row is now NaN (not a number) because it cannot calculate the percent change on the first row!

Get used to seeing NaN! We will have a whole section dedicated to dealing with NaN.

What do you think the output will look like when we do pct_change(3)?

In [31]:
df_pct_change_3 = df_spy.pct_change(3)
print(df_pct_change_3)

Price       Adj Close     Close      High       Low      Open    Volume
Date                                                                   
2005-01-03        NaN       NaN       NaN       NaN       NaN       NaN
2005-01-04        NaN       NaN       NaN       NaN       NaN       NaN
2005-01-05        NaN       NaN       NaN       NaN       NaN       NaN
2005-01-06  -0.014048 -0.014048 -0.021436 -0.013678 -0.025666 -0.142306
2005-01-07  -0.003282 -0.003282 -0.010868 -0.002617 -0.012369 -0.192574
...               ...       ...       ...       ...       ...       ...
2024-11-22   0.008826  0.008826  0.008646  0.015616  0.015307 -0.226374
2024-11-25   0.011905  0.011905  0.017045  0.018080  0.015482 -0.151725
2024-11-26   0.011758  0.011757  0.010435  0.018078  0.009100 -0.024150
2024-11-27   0.005575  0.005575  0.007884  0.006963  0.011454 -0.110557
2024-11-29   0.008401  0.008401  0.004144  0.007023  0.000233 -0.288963

[5012 rows x 6 columns]


Notice that now we have 3 rows of NaN!
That should not be a surprise since we are doing a percent change vs. the value from 3 rows up.



---



### 1.6 Working With Subsections of a DataFrame

Thus far, we have worked with the entire dataframe. But often, you might only want to work with specific columns or specific date ranges.
For example, maybe you don't care about doing a percent change on the volume column and only wanted to do it on the Adj Close column.

Let's start with these cases.




#### 1.6.1 Isolating One Column

Let's say we wanted to work with only the adjusted closing price column. To do this, we would enclose the target column name in a list next to the dataframe name.

In [32]:
adj_close_mean = df_spy['Adj Close'].mean()
print(adj_close_mean)
print(type(adj_close_mean))

202.97608900412692
<class 'numpy.float64'>


Notice how it comes out! We get just the answer as a float, not as a dataframe or a series.

What if you wanted it as a dataframe or a series?

Enclosing the column label in 2 sets of square brackets will return a series. We can then convert this series to a dataframe.

In [33]:
df_adj_close_mean = df_spy[['Adj Close']].mean()
print(df_adj_close_mean)

Price
Adj Close    202.976089
dtype: float64


In [34]:
# Convert a series to a dataframe
df_adj_close_mean = df_adj_close_mean.to_frame(name='spy Mean')
print(df_adj_close_mean)

             spy Mean
Price                
Adj Close  202.976089


the method `.to_frame()` will convert a series to a dataframe.
You can use the name parameter in the parentheses so that there will be a column label and not just 0.

#### 1.6.2 Selecting Multiple Columns
In this case, we want to isolate the Close and Adj Close columns.
We will use a similar syntax.


In [35]:
# 1) Isolate the columns of interest
df_subset = df_spy[['Close', 'Adj Close']]
print(df_subset)
# You will see the output has our two columns!
# Note that the column names you want to isolate are enclosed in separate sets of quotes

Price            Close   Adj Close
Date                              
2005-01-03  120.300003   80.973534
2005-01-04  118.830002   79.984070
2005-01-05  118.010002   79.432137
2005-01-06  118.610001   79.836021
2005-01-07  118.440002   79.721573
...                ...         ...
2024-11-22  595.510010  582.077271
2024-11-25  597.530029  584.051575
2024-11-26  600.650024  587.101318
2024-11-27  598.830017  585.322327
2024-11-29  602.549988  588.958374

[5012 rows x 2 columns]


In [36]:
# 2) Apply a method
df_subset_mean = df_subset.mean()
df_subset_mean

,0
Price,
Close,233.659220
Adj Close,202.976089


Note that the output moves the columns as rows and returns a series.
You could convert this to a dataframe if you wanted to.

This works for all other methods.

Let's try the pct_change

In [37]:
df_subset_pct_change = df_subset.pct_change()
df_subset_pct_change

Price,Close,Adj Close
Date,,
2005-01-03,NaN,NaN
2005-01-04,-0.012219,-0.012220
2005-01-05,-0.006901,-0.006901
2005-01-06,0.005084,0.005085
2005-01-07,-0.001433,-0.001434
...,...,...
2024-11-22,0.003099,0.003100
2024-11-25,0.003392,0.003392
2024-11-26,0.005221,0.005222


This returns a dataframe as expected.
But what if you wanted to add the percent change of the Adj Close as a new column instead of a new dataframe?

##### 1.63 Adding a New Column

To add the pct_change of the Adj Close price, you assign the method to the column of interest. In this case, we will calculate the daily return of spy and add it as a new column called 'Daily % Change'
Pay attention to the syntax.

In [38]:
df_spy['Daily % Change'] = df_spy['Adj Close'].pct_change()
df_spy

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change
Date,,,,,,,
2005-01-03,80.973534,120.300003,121.760002,119.900002,121.559998,55748000,NaN
2005-01-04,79.984070,118.830002,120.540001,118.440002,120.459999,69167600,-0.012220
2005-01-05,79.432137,118.010002,119.250000,118.000000,118.739998,65667300,-0.006901
2005-01-06,79.836021,118.610001,119.150002,118.260002,118.440002,47814700,0.005085
2005-01-07,79.721573,118.440002,119.230003,118.129997,118.970001,55847700,-0.001434
...,...,...,...,...,...,...,...
2024-11-22,582.077271,595.510010,596.150024,593.150024,593.659973,38226400,0.003100
2024-11-25,584.051575,597.530029,600.859985,595.200012,599.520020,42441400,0.003392
2024-11-26,587.101318,600.650024,601.330017,598.070007,598.799988,45621300,0.005222


Check df_spy.info() to see the changes

In [39]:
df_spy.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5012 entries, 2005-01-03 to 2024-11-29
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Adj Close       5012 non-null   float64
 1   Close           5012 non-null   float64
 2   High            5012 non-null   float64
 3   Low             5012 non-null   float64
 4   Open            5012 non-null   float64
 5   Volume          5012 non-null   int64  
 6   Daily % Change  5011 non-null   float64
dtypes: float64(6), int64(1)
memory usage: 313.2 KB


This is a great example to show how to not only add a column, but to inspect the results afterwardws with .info()!
Notice that we have the new column listed at the end, but that also, we are missing a value, or it shows we have an NaN?
Look at the output and figure out how we were able to tell that!

### 1.7 Method Chaining

Method chaining is a great example of how amazingly simple Python can be.
Let's say we wanted to calculate the mean daily % change of the Adj Close column.

In [40]:
mean_pct_change = df_spy['Adj Close'].pct_change().mean()
mean_pct_change

np.float64(0.0004681635637536139)

And that's it! You just did one line of code instead of two.
And you can chain as many as you need to (until something breaks of course). Just remember that order matters, and you can always put things in parentheses if you need to.

---


### 1.8 Other Important Methods

We will now look at some other important methods:
1. `min()` and `max()`
2. `round()`
3. `argmax()` and `argmin()`
4. `idxmax()` and `idxmin()`


##### 1.8.1 `max()` and `min()`

**What They Do:**

- min(): Returns the minimum value in the DataFrame or Series.
- max(): Returns the maximum value in the DataFrame or Series.

**Why They’re Useful:**

- Helps find the lowest and highest stock prices, returns, or other metrics.
- Essential for identifying extreme values (e.g., yearly lows or highs).

In [41]:
print('Maximum Values:')
print(df_spy.max())
print('\nMinimum Values:')
print(df_spy.min())

Maximum Values:
Price
Adj Close         5.889584e+02
Close             6.025500e+02
High              6.033500e+02
Low               5.993800e+02
Open              6.004600e+02
Volume            8.710263e+08
Daily % Change    1.451979e-01
dtype: float64

Minimum Values:
Price
Adj Close         4.955752e+01
Close             6.811000e+01
High              7.000000e+01
Low               6.710000e+01
Open              6.795000e+01
Volume            1.527000e+07
Daily % Change   -1.094239e-01
dtype: float64


##### 1.8.2 `round()`

**What It Does:**
- Rounds numerical values to a specified number of decimal places.

**Why It’s Useful:**
- Ensures consistent formatting for financial data (e.g., prices to 2 decimal places).
- Pandas makes it very easy to round! You just need to put the number of decimal places inside the parentheses.


In [42]:
# Round to 2 decimal places
print("Rounded values:\n", df_spy.head().round(2))

# Notice the method chaining!

Rounded values:
 Price       Adj Close   Close    High     Low    Open    Volume  \
Date                                                              
2005-01-03      80.97  120.30  121.76  119.90  121.56  55748000   
2005-01-04      79.98  118.83  120.54  118.44  120.46  69167600   
2005-01-05      79.43  118.01  119.25  118.00  118.74  65667300   
2005-01-06      79.84  118.61  119.15  118.26  118.44  47814700   
2005-01-07      79.72  118.44  119.23  118.13  118.97  55847700   

Price       Daily % Change  
Date                        
2005-01-03             NaN  
2005-01-04           -0.01  
2005-01-05           -0.01  
2005-01-06            0.01  
2005-01-07           -0.00  


##### 1.8.3 `argmax()` and `argmin()`

**What They Do:**
- argmax(): Returns the position (index) of the maximum value in a Series.
- argmin(): Returns the position (index) of the minimum value in a Series.

**Why They’re Useful:**
- Identifies where the highest or lowest values occur.
- Helps find peaks and troughs in time series data.


In [43]:
# Find the index position of the highest and lowest adjusted closing prices
print("Position of max value in spy:", df_spy['Adj Close'].argmax())
print("Position of min value in spy:", df_spy['Adj Close'].argmin())


Position of max value in spy: 5011
Position of min value in spy: 1051


The output here is simply telling you index position 0 (row 0) has the lowest price while index position 5011 (row 5011, starting at 0) has the higest value.

But what if we wanted the datetime index values that correspond to those integer locations?

##### 1.8.4 `idxmax()` and `idxmin()`

**What They Do:**
- idxmax(): Returns the index label of the maximum value.
- idxmin(): Returns the index label of the minimum value.

**Why They’re Useful:**
- Pinpoints the date or identifier associated with extreme values in time series data.


In [44]:
print(f"Date of lowest price: {df_spy['Adj Close'].idxmin()}")
print(f"Date of higest price: {df_spy['Adj Close'].idxmax()}")

Date of lowest price: 2009-03-09 00:00:00
Date of higest price: 2024-11-29 00:00:00




---



### 1.9 Using .iloc, .loc and .at for DataFrame Slicing

*This is one of the most important topics in this course, and should be practiced often!*

**Consider this section to be a more powerful Excel equivalent of VLOOKUP.**


Efficiently slicing and accessing data is a cornerstone of financial data analysis. Financial datasets, such as stock prices, returns, and volumes, often span large time periods and involve multiple variables. The ability to extract specific subsets of data—whether by position, label, or exact value—allows analysts to focus on relevant information, perform precise calculations, and identify key trends. Methods like .iloc, .loc, and .at empower users to navigate complex datasets with ease, making it possible to filter by date ranges, isolate specific securities, and retrieve individual values. Mastering these tools is essential for tasks such as portfolio analysis, strategy backtesting, and financial reporting, where accuracy and efficiency are paramount.








##### 1.9.1 `.iloc`
**What It Does**
- `.iloc` is used for integer-based indexing in a DataFrame.
- Allows you to select rows and columns by their numerical positions.

**How to Use It**
1. Select Specific Rows and Columns:
- df.iloc[row, column]
2. Slice Rows or Columns:
- Use : for slicing, e.g., df.iloc[start:stop, :].


In [45]:
# First row and all columns; notice that it produces a series (we only have one row...)
print(df_spy.iloc[0, :])
print(type(df_spy.iloc[0, :]))

Price
Adj Close         8.097353e+01
Close             1.203000e+02
High              1.217600e+02
Low               1.199000e+02
Open              1.215600e+02
Volume            5.574800e+07
Daily % Change             NaN
Name: 2005-01-03 00:00:00, dtype: float64
<class 'pandas.core.series.Series'>


In [46]:
# First 5 rows of Adj Close
print(df_spy.iloc[0:5, 0])


Date
2005-01-03    80.973534
2005-01-04    79.984070
2005-01-05    79.432137
2005-01-06    79.836021
2005-01-07    79.721573
Name: Adj Close, dtype: float64


In [47]:
# Last 5 rows
print(df_spy.iloc[-5:, :])


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2024-11-22  582.077271  595.510010  596.150024  593.150024  593.659973   
2024-11-25  584.051575  597.530029  600.859985  595.200012  599.520020   
2024-11-26  587.101318  600.650024  601.330017  598.070007  598.799988   
2024-11-27  585.322327  598.830017  600.849976  597.280029  600.460022   
2024-11-29  588.958374  602.549988  603.349976  599.380005  599.659973   

Price         Volume  Daily % Change  
Date                                  
2024-11-22  38226400        0.003100  
2024-11-25  42441400        0.003392  
2024-11-26  45621300        0.005222  
2024-11-27  34000200       -0.003030  
2024-11-29  30177400        0.006212  


In [48]:
# Last 5 rows, columns 2 and 3 - Remember 0 indexing!
print(df_spy.iloc[-5:, 1:3])


Price            Close        High
Date                              
2024-11-22  595.510010  596.150024
2024-11-25  597.530029  600.859985
2024-11-26  600.650024  601.330017
2024-11-27  598.830017  600.849976
2024-11-29  602.549988  603.349976


#### 1.9.2 `.loc`: Label Based Indexing

**What It Does**
- .loc is used for label-based indexing in a DataFrame.
- Allows you to select rows and columns by their labels (e.g., dates or column names).

**How to Use It**
1. Select Specific Rows and Columns by Label:
- df.loc[row_label, column_label]
2. Slice Rows and Columns by Label:
- df.loc[start_row:end_row, start_col:end_col]

**Why It’s Useful**
- Works naturally with labeled indices like dates or named columns.
- Ideal for slicing time series data or filtering specific columns.

In [49]:
# Data for a specific date
print(df_spy.loc['2023-01-03'])


Price
Adj Close         3.632320e+02
Close             3.808200e+02
High              3.864300e+02
Low               3.778300e+02
Open              3.843700e+02
Volume            7.485070e+07
Daily % Change   -4.209758e-03
Name: 2023-01-03 00:00:00, dtype: float64


In [50]:
# Data for January 2023
print(df_spy.loc['2023-01-01':'2023-01-31'].head(5))


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2023-01-03  363.232025  380.820007  386.429993  377.829987  384.369995   
2023-01-04  366.036255  383.760010  385.880005  380.000000  383.179993   
2023-01-05  361.858490  379.380005  381.839996  378.760010  381.720001   
2023-01-06  370.156738  388.079987  389.250000  379.410004  382.609985   
2023-01-09  369.946808  387.859985  393.700012  387.670013  390.369995   

Price          Volume  Daily % Change  
Date                                   
2023-01-03   74850700       -0.004210  
2023-01-04   85934100        0.007720  
2023-01-05   76970500       -0.011414  
2023-01-06  104189600        0.022932  
2023-01-09   73978100       -0.000567  


In [51]:
# Slices can be saved as new dataframes
df_spy_jan = df_spy.loc['2023-01-01':'2023-01-31']
df_spy_jan.head()

# Notice how we didn't specify columns here, so it returns all of them

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change
Date,,,,,,,
2023-01-03,363.232025,380.820007,386.429993,377.829987,384.369995,74850700,-0.004210
2023-01-04,366.036255,383.760010,385.880005,380.000000,383.179993,85934100,0.007720
2023-01-05,361.858490,379.380005,381.839996,378.760010,381.720001,76970500,-0.011414
2023-01-06,370.156738,388.079987,389.250000,379.410004,382.609985,104189600,0.022932
2023-01-09,369.946808,387.859985,393.700012,387.670013,390.369995,73978100,-0.000567


In [52]:
# Keeping only specific columns by labels; notice the column labels are in a list.
df_spy_jan = df_spy.loc['2023-01-01':'2023-01-31', ['Adj Close', 'Volume']]
print(df_spy_jan.head())

Price        Adj Close     Volume
Date                             
2023-01-03  363.232025   74850700
2023-01-04  366.036255   85934100
2023-01-05  361.858490   76970500
2023-01-06  370.156738  104189600
2023-01-09  369.946808   73978100


##### 1.9.3 `.at`

**What It Does**

- .at is used for fast access to a single scalar value by label.
- It is optimized for quick retrieval and is faster than .loc for single values.

**How to Use It**
1. Access a Single Value:
 - df.at[row_label, column_label]

**Why It’s Useful**
- Highly efficient for retrieving a single value.
- Best used when performance is critical for single value lookups.

**One of the main advantages of using `.at` is that it returns the value and not a dataframe or a series!
Using `.loc` or `.iloc` can return either a dataframe or a series, and this could cause errors in functions when you are trying to access a single value. To avoid ambiguity and uncertainty of what `.iloc` and `.loc` will return when all you want is a single value, use `.at`**

In [53]:
# Adjusted close price on a specific date
print(df_spy.at['2023-01-03', 'Adj Close'])


363.2320251464844



---
### 1.10 Filtering DataFrames Based on Conditions

Filtering DataFrames is a fundamental skill for financial analysis, allowing you to extract and work with subsets of data that meet specific criteria. For example, you might want to isolate days where a stock's adjusted closing price exceeded a certain value or identify high-volume trading days. By defining conditions such as >, <, >=, and combining multiple conditions with logical operators (&, |, ~), you can quickly narrow your focus to the most relevant data for your analysis.

At the heart of filtering is the concept of a Boolean mask. A Boolean mask is a series of True or False values generated by evaluating a condition on a DataFrame or Series. This mask is used to filter rows where the condition is True, effectively creating a new subset of data. Boolean masks are crucial because they make the filtering process efficient and intuitive, allowing you to apply conditions directly to your data without requiring complex loops or manual selection.

Whether you're identifying trends, evaluating trading strategies, or preparing data for visualization, mastering filtering ensures that you can work with clean, targeted datasets tailored to your needs. This section will demonstrate how to filter data based on single and multiple conditions, and how to refine the results to include only specific columns.


#### 1.10.1 What is a Boolean Mask?

The best way to explain the mask is to demonstrate how it works.

Let's say you want to extract all days from df_spy when the Daily % Change is greater than 1%.

You would start with this:

In [54]:
# Boolean mask, all days where Daily % Change is greater than 1%
df_spy['Daily % Change'] > 0.01

,Daily % Change
Date,
2005-01-03,False
2005-01-04,False
2005-01-05,False
2005-01-06,False
2005-01-07,False
...,...
2024-11-22,False
2024-11-25,False
2024-11-26,False


Notice that it produces a dataframe that just returned the Daily % Change column, but with no values. It just tells you if each row satisfied the condition. But what if you want to retaill ALL columns and ALL values?

#### 1.10.2 Filtering Rows Based on a Single Condition

Let's look at the example of filtering all days where the Adj Close is > 150.

In [55]:
# Filter rows where Adj Close is greater than 150
filtered_df = df_spy[df_spy['Adj Close'] > 150]
print(filtered_df)


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2014-02-27  150.175110  185.820007  185.869995  184.369995  184.580002   
2014-02-28  150.554962  186.289993  187.149994  185.050003  185.789993   
2014-03-04  151.597519  187.580002  187.979996  186.750000  186.789993   
2014-03-05  151.734863  187.750000  188.070007  187.449997  187.740005   
2014-03-06  152.082382  188.179993  188.610001  187.779999  188.210007   
...                ...         ...         ...         ...         ...   
2024-11-22  582.077271  595.510010  596.150024  593.150024  593.659973   
2024-11-25  584.051575  597.530029  600.859985  595.200012  599.520020   
2024-11-26  587.101318  600.650024  601.330017  598.070007  598.799988   
2024-11-27  585.322327  598.830017  600.849976  597.280029  600.460022   
2024-11-29  588.958374  602.549988  603.349976  599.380005  599.659973   

Price          Volume  Daily % Change

Explanation:

- The condition df_spy['Adj Close'] > 150 generates a boolean mask (True or False for each row).

- Passing this mask to df_spy returns only rows where the condition is True.

**Key Point:**
Please remember the syntax for when you want to filter:

`df[df[column name] condition]`

Note that the df name appears twice! Once to create the boolean mask, the second time to retain retrieve the data from the df based on the boolean mask.

#### 1.10.3 Filtering Rows Where Adj Close >= 150

Let's do the same thing but use the >= operator

In [56]:
# Filter rows where Adj Close is greater than or equal to 150
filtered_df = df_spy[df_spy['Adj Close'] >= 150]
print(filtered_df)


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2014-02-27  150.175110  185.820007  185.869995  184.369995  184.580002   
2014-02-28  150.554962  186.289993  187.149994  185.050003  185.789993   
2014-03-04  151.597519  187.580002  187.979996  186.750000  186.789993   
2014-03-05  151.734863  187.750000  188.070007  187.449997  187.740005   
2014-03-06  152.082382  188.179993  188.610001  187.779999  188.210007   
...                ...         ...         ...         ...         ...   
2024-11-22  582.077271  595.510010  596.150024  593.150024  593.659973   
2024-11-25  584.051575  597.530029  600.859985  595.200012  599.520020   
2024-11-26  587.101318  600.650024  601.330017  598.070007  598.799988   
2024-11-27  585.322327  598.830017  600.849976  597.280029  600.460022   
2024-11-29  588.958374  602.549988  603.349976  599.380005  599.659973   

Price          Volume  Daily % Change

The output is the same since there were no days where the Adj Close was exactly 150.

Let's try filtering on a different column:

#### 1.10.4 Filtering Based on Another Column


In [57]:
# Filter rows where Volume is greater than 1,000,000
filtered_df = df_spy[df_spy['Volume'] > 1_000_000]
print(filtered_df)


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2005-01-03   80.973534  120.300003  121.760002  119.900002  121.559998   
2005-01-04   79.984070  118.830002  120.540001  118.440002  120.459999   
2005-01-05   79.432137  118.010002  119.250000  118.000000  118.739998   
2005-01-06   79.836021  118.610001  119.150002  118.260002  118.440002   
2005-01-07   79.721573  118.440002  119.230003  118.129997  118.970001   
...                ...         ...         ...         ...         ...   
2024-11-22  582.077271  595.510010  596.150024  593.150024  593.659973   
2024-11-25  584.051575  597.530029  600.859985  595.200012  599.520020   
2024-11-26  587.101318  600.650024  601.330017  598.070007  598.799988   
2024-11-27  585.322327  598.830017  600.849976  597.280029  600.460022   
2024-11-29  588.958374  602.549988  603.349976  599.380005  599.659973   

Price         Volume  Daily % Change 

Nothing that different here, as the process is the same.

#### 1.10.5 Filtering with Multiple Conditions

**Using & (AND Condition):**

Filter rows where Adj Close > 150 and Volume > 1,000,000:

In [58]:
# Filter rows with multiple conditions
filtered_df = df_spy[(df_spy['Adj Close'] > 150) & (df_spy['Volume'] > 1_000_000)]
print(filtered_df)

# Again, notice that each condition is enclosed in separate parentheses!

Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2014-02-27  150.175110  185.820007  185.869995  184.369995  184.580002   
2014-02-28  150.554962  186.289993  187.149994  185.050003  185.789993   
2014-03-04  151.597519  187.580002  187.979996  186.750000  186.789993   
2014-03-05  151.734863  187.750000  188.070007  187.449997  187.740005   
2014-03-06  152.082382  188.179993  188.610001  187.779999  188.210007   
...                ...         ...         ...         ...         ...   
2024-11-22  582.077271  595.510010  596.150024  593.150024  593.659973   
2024-11-25  584.051575  597.530029  600.859985  595.200012  599.520020   
2024-11-26  587.101318  600.650024  601.330017  598.070007  598.799988   
2024-11-27  585.322327  598.830017  600.849976  597.280029  600.460022   
2024-11-29  588.958374  602.549988  603.349976  599.380005  599.659973   

Price          Volume  Daily % Change

Explanation:

- Wrap each condition in parentheses () to avoid errors.
- Combine conditions with & for "AND" logic.

**Using | (OR Condition):**

Filter rows where Adj Close > 150 or Volume > 1,000,000:

In [59]:
# Filter rows with OR condition
filtered_df = df_spy[(df_spy['Adj Close'] > 150) | (df_spy['Volume'] > 1_000_000)]
print(filtered_df)


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2005-01-03   80.973534  120.300003  121.760002  119.900002  121.559998   
2005-01-04   79.984070  118.830002  120.540001  118.440002  120.459999   
2005-01-05   79.432137  118.010002  119.250000  118.000000  118.739998   
2005-01-06   79.836021  118.610001  119.150002  118.260002  118.440002   
2005-01-07   79.721573  118.440002  119.230003  118.129997  118.970001   
...                ...         ...         ...         ...         ...   
2024-11-22  582.077271  595.510010  596.150024  593.150024  593.659973   
2024-11-25  584.051575  597.530029  600.859985  595.200012  599.520020   
2024-11-26  587.101318  600.650024  601.330017  598.070007  598.799988   
2024-11-27  585.322327  598.830017  600.849976  597.280029  600.460022   
2024-11-29  588.958374  602.549988  603.349976  599.380005  599.659973   

Price         Volume  Daily % Change 

#### 1.10.6 Negating Conditions with `~`

Filter rows where Adj Close is *not* greater than 150:

In [60]:
# Negating a condition
filtered_df = df_spy[~(df_spy['Adj Close'] > 150)]
print(filtered_df)

# Look at how efficient this can be!

Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2005-01-03   80.973534  120.300003  121.760002  119.900002  121.559998   
2005-01-04   79.984070  118.830002  120.540001  118.440002  120.459999   
2005-01-05   79.432137  118.010002  119.250000  118.000000  118.739998   
2005-01-06   79.836021  118.610001  119.150002  118.260002  118.440002   
2005-01-07   79.721573  118.440002  119.230003  118.129997  118.970001   
...                ...         ...         ...         ...         ...   
2014-04-07  149.636490  184.339996  186.259995  183.960007  185.949997   
2014-04-10  148.678619  183.160004  187.169998  182.929993  187.080002   
2014-04-11  147.339294  181.509995  183.419998  181.309998  182.139999   
2014-04-14  148.500076  182.940002  183.369995  181.440002  182.929993   
2014-04-15  149.522873  184.199997  184.330002  181.509995  183.320007   

Price          Volume  Daily % Change

#### 1.10.7 Keeping Only Specific Columns After Filtering
So far, we have kept all columns after filtering. But what if we wanted to filter based on a condition, yet keep only some of the columns?

Example: Keep Only Adj Close and Volume

After filtering, you can select specific columns by chaining [['col1', 'col2']]:

In [61]:
# Filter rows and keep only Adj Close and Volume
filtered_df = df_spy[df_spy['Adj Close'] > 150][['Adj Close', 'Volume']]
print(filtered_df)


Price        Adj Close     Volume
Date                             
2014-02-27  150.175110   93880800
2014-02-28  150.554962  150842000
2014-03-04  151.597519  167545900
2014-03-05  151.734863   88376900
2014-03-06  152.082382   82516500
...                ...        ...
2024-11-22  582.077271   38226400
2024-11-25  584.051575   42441400
2024-11-26  587.101318   45621300
2024-11-27  585.322327   34000200
2024-11-29  588.958374   30177400

[2700 rows x 2 columns]


Example: Multiple Conditions with Selected Columns

Filter rows where Adj Close > 150 and Volume > 1,000,000, and keep only
Adj Close and Daily % Change:

In [62]:
# Filter rows and keep specific columns
filtered_df = df_spy[(df_spy['Adj Close'] > 150) & (df_spy['Volume'] > 1_000_000)][['Adj Close', 'Daily % Change']]
print(filtered_df)


Price        Adj Close  Daily % Change
Date                                  
2014-02-27  150.175110        0.005248
2014-02-28  150.554962        0.002529
2014-03-04  151.597519        0.014056
2014-03-05  151.734863        0.000906
2014-03-06  152.082382        0.002290
...                ...             ...
2024-11-22  582.077271        0.003100
2024-11-25  584.051575        0.003392
2024-11-26  587.101318        0.005222
2024-11-27  585.322327       -0.003030
2024-11-29  588.958374        0.006212

[2700 rows x 2 columns]


#### 1.10.8 Summary

1. Chaining Conditions:

- Always use parentheses () around each condition.
- Combine conditions with & (AND), | (OR), and ~ (NOT).

2. Extracting Specific Columns:

- Use [['col1', 'col2']] to refine the filtered DataFrame to include only relevant columns.

3. Boolean Masks:

- Conditions generate boolean masks that filter rows efficiently.

4. Avoid Common Errors:

- Do not use Python's and/or operators. Instead, use Pandas-specific &/| for element-wise comparisons.

**In filtering, most errors occur because of bad syntax and not placing the parentheses and square brackets at the correct places.**

#### 1.10.9 Common Uses and Examples

This section will present you with some common uses of filtering with time series data.



1. Identifying days with high volume

In [63]:
high_volume_df = df_spy[df_spy['Volume'] > 50_000_000]
high_volume_df

# Python Trick: you can use underscores in really large numbers as a thousand separator to make it more
# visually applealing.

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change
Date,,,,,,,
2005-01-03,80.973534,120.300003,121.760002,119.900002,121.559998,55748000,NaN
2005-01-04,79.984070,118.830002,120.540001,118.440002,120.459999,69167600,-0.012220
2005-01-05,79.432137,118.010002,119.250000,118.000000,118.739998,65667300,-0.006901
2005-01-07,79.721573,118.440002,119.230003,118.129997,118.970001,55847700,-0.001434
2005-01-10,80.098518,119.000000,119.459999,118.339996,118.339996,56563300,0.004728
...,...,...,...,...,...,...,...
2024-10-15,566.702026,579.780029,584.900024,578.539978,584.590027,54203600,-0.007770
2024-10-31,555.813416,568.640015,575.630005,568.440002,575.559998,60182500,-0.019603
2024-11-06,577.707947,591.039978,591.929993,585.390015,589.200012,68182000,0.024865


2. Filter Stocks with Significant Daily Changes:

In [64]:
big_changes_df = df_spy[abs(df_spy['Daily % Change']) > 0.02]
big_changes_df

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change
Date,,,,,,,
2006-06-15,86.731674,126.120003,126.360001,123.860001,123.949997,134057000,0.021215
2006-06-29,87.909393,127.269997,127.349998,125.169998,125.199997,110634800,0.020201
2007-02-27,97.320023,139.500000,144.199997,139.000000,143.880005,274466500,-0.039058
2007-07-26,104.120255,148.020004,150.800003,146.389999,150.190002,467592500,-0.023679
2007-08-03,101.151855,143.800003,147.580002,143.199997,147.279999,359398200,-0.025745
...,...,...,...,...,...,...,...
2024-07-24,527.403809,541.229980,549.169983,540.289978,548.859985,74515300,-0.022663
2024-08-05,504.163116,517.380005,523.580017,510.269989,511.640015,146267400,-0.029124
2024-08-08,517.094238,530.650024,531.289978,521.840027,523.909973,63276600,0.023117


3. Refine Analysis by Selecting Columns:

In [65]:
specific_df = df_spy[(df_spy['Adj Close'] > 150) & (df_spy['Volume'] > 1_000_000)][['Adj Close', 'Volume']]
specific_df

Price,Adj Close,Volume
Date,,
2014-02-27,150.175110,93880800
2014-02-28,150.554962,150842000
2014-03-04,151.597519,167545900
2014-03-05,151.734863,88376900
2014-03-06,152.082382,82516500
...,...,...
2024-11-22,582.077271,38226400
2024-11-25,584.051575,42441400
2024-11-26,587.101318,45621300


These are just some exmaples of how we will use filters. We will also use them in conjunction with `.loc` and `.iloc` to help filter data between specific dates.

Imagine this scenario:
You are a quantitative reasearcher, and you want to know how many times in each calendar year since 2000 that SPY had at least a -10% correction.

How would you do that? You will need to use multiple filtering methods and method chaining.

This is the kind of task that you will learn how to do in our courses.

It is time to move onto one of the most important concepts that you need to master with dataframes: Vectorization.

### 1.11 Introduction to Vectorization in Pandas

Vectorization is one of the most powerful concepts in data analysis, allowing operations to be applied across entire datasets at once, rather than iterating through individual elements. This approach leverages the optimized, low-level implementation of operations in libraries like Pandas and NumPy, resulting in significant speed improvements and more concise, readable code.

**Why Vectorization Is Important**

1. Speed:

- Vectorized operations are highly optimized and can be orders of magnitude faster than equivalent operations using Python loops.
- They take advantage of compiled C code under the hood, avoiding the overhead of Python's interpreted loops.

2. Simplicity:

- Code using vectorization is more concise, easier to read, and less error-prone compared to loops.
- A single line of vectorized code can replace many lines of loop-based logic.

3. Scalability:

- Vectorized operations scale well to large datasets, which is critical in finance when dealing with millions of rows of time-series or trade data.

In financial analysis, where time and accuracy are paramount, understanding and applying vectorization ensures your code remains efficient and professional.

We will continue to work with `df_spy` and show some common operations.

#### 1.11.1 Adding, Subtracting, Multiplying, Dividing Entire Columns

Perform arithmetic operations across entire columns.

In [66]:
# Calculate a 10% increase in Adj Close prices
df_spy['Adj Close + 10%'] = df_spy['Adj Close'] * 1.10

# Calculate the difference between High and Low prices
df_spy['High - Low'] = df_spy['High'] - df_spy['Low']

# Convert Volume to millions
df_spy['Volume (M)'] = df_spy['Volume'] / 1_000_000

print(df_spy.tail().round(2))


Price       Adj Close   Close    High     Low    Open    Volume  \
Date                                                              
2024-11-22     582.08  595.51  596.15  593.15  593.66  38226400   
2024-11-25     584.05  597.53  600.86  595.20  599.52  42441400   
2024-11-26     587.10  600.65  601.33  598.07  598.80  45621300   
2024-11-27     585.32  598.83  600.85  597.28  600.46  34000200   
2024-11-29     588.96  602.55  603.35  599.38  599.66  30177400   

Price       Daily % Change  Adj Close + 10%  High - Low  Volume (M)  
Date                                                                 
2024-11-22            0.00           640.28        3.00       38.23  
2024-11-25            0.00           642.46        5.66       42.44  
2024-11-26            0.01           645.81        3.26       45.62  
2024-11-27           -0.00           643.85        3.57       34.00  
2024-11-29            0.01           647.85        3.97       30.18  


I hope you see the power of this. We were able to execute operations across all rows for specified columns in one line of code.

The alternative would be to use a for loop, going through each row and applying the operation one at a time! You must get used to vectorization if you want to have efficient and professional code.

#### 1.11.2 Applying Mathematical Functions

While we haven't discussed the numpy library yet, we will use a couple of common methods for demonstration purposes.

- Use functions like np.log(), np.sqrt(), or Pandas .abs() for transformations.

In [67]:
import numpy as np

# Logarithm of Adj Close
df_spy['Log(Adj Close)'] = np.log(df_spy['Adj Close'])

# Absolute values of Daily % Change
df_spy['Abs(Daily % Change)'] = df_spy['Daily % Change'].abs()

# Show the tail of the last last 4 columns
print(df_spy.iloc[:, -4:].tail().round(2))

Price       High - Low  Volume (M)  Log(Adj Close)  Abs(Daily % Change)
Date                                                                   
2024-11-22        3.00       38.23            6.37                 0.00
2024-11-25        5.66       42.44            6.37                 0.00
2024-11-26        3.26       45.62            6.38                 0.01
2024-11-27        3.57       34.00            6.37                 0.00
2024-11-29        3.97       30.18            6.38                 0.01


Once again you see how easy it is to add a new column to our dataframe based on a mathematical function or transformation of our existing data.

You could of course create a new dataframe instead of adding new columns.

#### 1.11.3 Boolean Filtering with Vectorized Conditions

This is exactly what we did before for filtering, but we are showing you how that in itself is a vectorization operation!

Look how fast these operations are performed!

In [68]:
# Identify rows where Adj Close > 150 and Volume > 10 million
high_value_days = df_spy[(df_spy['Adj Close'] > 150) & (df_spy['Volume'] > 10_000_000)]
high_value_days

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change)
Date,,,,,,,,,,,,
2014-02-27,150.175110,185.820007,185.869995,184.369995,184.580002,93880800,0.005248,165.192621,1.500000,93.8808,5.011802,0.005248
2014-02-28,150.554962,186.289993,187.149994,185.050003,185.789993,150842000,0.002529,165.610458,2.099991,150.8420,5.014328,0.002529
2014-03-04,151.597519,187.580002,187.979996,186.750000,186.789993,167545900,0.014056,166.757271,1.229996,167.5459,5.021229,0.014056
2014-03-05,151.734863,187.750000,188.070007,187.449997,187.740005,88376900,0.000906,166.908350,0.620010,88.3769,5.022135,0.000906
2014-03-06,152.082382,188.179993,188.610001,187.779999,188.210007,82516500,0.002290,167.290620,0.830002,82.5165,5.024422,0.002290
...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-22,582.077271,595.510010,596.150024,593.150024,593.659973,38226400,0.003100,640.284998,3.000000,38.2264,6.366603,0.003100
2024-11-25,584.051575,597.530029,600.859985,595.200012,599.520020,42441400,0.003392,642.456732,5.659973,42.4414,6.369989,0.003392
2024-11-26,587.101318,600.650024,601.330017,598.070007,598.799988,45621300,0.005222,645.811450,3.260010,45.6213,6.375197,0.005222


#### 1.11.4 Conditional Assignment

Again, while we haven't walked through numpy yet, here we will show you the Python equivalent of the Excel If statement:

This is the excel statement we all know and love:

`IF(condition to evaluate, output if true, output if false)`

In Python, we use np.where:

`np.where(condition to evaluate, output if true, output if false)`

In [69]:
# Add a column marking whether Adj Close > 150
df_spy['Above 150'] = np.where(df_spy['Adj Close'] > 150, 'Yes', 'No')
df_spy.tail()

# Look at the last column; You will see either Yes or No. You could then filter based on the Yes or No condition!

Price,Adj Close,Close,High,Low,Open,Volume,Daily % Change,Adj Close + 10%,High - Low,Volume (M),Log(Adj Close),Abs(Daily % Change),Above 150
Date,,,,,,,,,,,,,
2024-11-22,582.077271,595.510010,596.150024,593.150024,593.659973,38226400,0.003100,640.284998,3.000000,38.2264,6.366603,0.003100,Yes
2024-11-25,584.051575,597.530029,600.859985,595.200012,599.520020,42441400,0.003392,642.456732,5.659973,42.4414,6.369989,0.003392,Yes
2024-11-26,587.101318,600.650024,601.330017,598.070007,598.799988,45621300,0.005222,645.811450,3.260010,45.6213,6.375197,0.005222,Yes
2024-11-27,585.322327,598.830017,600.849976,597.280029,600.460022,34000200,-0.003030,643.854559,3.569946,34.0002,6.372163,0.003030,Yes
2024-11-29,588.958374,602.549988,603.349976,599.380005,599.659973,30177400,0.006212,647.854211,3.969971,30.1774,6.378356,0.006212,Yes


#### 1.11.5 Using `.sub`, `.add`, `.mul`, and `.div` in Pandas

Pandas provides arithmetic methods like `.sub()`, `.add()`, `.mul()`, and `.div()` as alternatives to mathematical operators (-, +, *, /).

These methods offer additional flexibility, such as handling mismatched indices and applying operations with specific alignment.

**When to Use Arithmetic Methods Instead of Operators**
1. Index Alignment:

- These methods align rows and columns based on their labels, ensuring accurate operations even with mismatched indices or column names.

2. Adding Fill Values for Missing Data:

- They allow you to specify a fill_value to handle missing data, which is not possible with basic operators.

3. Clarity:

- Using named methods can make your intentions clearer in more complex operations.

---

Basic Arithmetic Using Methods

Perform arithmetic across columns in df_spy.

In [70]:
# Let's create a subset of df_appl so that we don't have a messy df with too many columns
# You will also learn a best-practices technique to create a new df with a subset of columns from the original

columns_to_keep = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
df_spy_copy = df_spy[columns_to_keep]  # notice how columns_to_keep is a list, which is required

# Subtract 'Low' from 'High' to calculate daily range
df_spy_copy['Range'] = df_spy_copy['High'].sub(df_spy_copy['Low'])

# Add 10% to 'Adj Close'
df_spy_copy['Adj Close + 10%'] = df_spy_copy['Adj Close'].mul(1.10)

# Divide 'Volume' by 1,000,000 to convert to millions
df_spy_copy['Volume (M)'] = df_spy_copy['Volume'].div(1_000_000)

# Add 'High' and 'Low' to calculate their sum
df_spy_copy['High + Low'] = df_spy_copy['High'].add(df_spy_copy['Low'])

df_spy_copy

/tmp/ipykernel_612/4007410661.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_spy_copy['Range'] = df_spy_copy['High'].sub(df_spy_copy['Low'])
/tmp/ipykernel_612/4007410661.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_spy_copy['Adj Close + 10%'] = df_spy_copy['Adj Close'].mul(1.10)
/tmp/ipykernel_612/4007410661.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the doc

Price,Open,High,Low,Close,Adj Close,Volume,Range,Adj Close + 10%,Volume (M),High + Low
Date,,,,,,,,,,
2005-01-03,121.559998,121.760002,119.900002,120.300003,80.973534,55748000,1.860001,89.070887,55.7480,241.660004
2005-01-04,120.459999,120.540001,118.440002,118.830002,79.984070,69167600,2.099998,87.982477,69.1676,238.980003
2005-01-05,118.739998,119.250000,118.000000,118.010002,79.432137,65667300,1.250000,87.375350,65.6673,237.250000
2005-01-06,118.440002,119.150002,118.260002,118.610001,79.836021,47814700,0.889999,87.819624,47.8147,237.410004
2005-01-07,118.970001,119.230003,118.129997,118.440002,79.721573,55847700,1.100006,87.693730,55.8477,237.360001
...,...,...,...,...,...,...,...,...,...,...
2024-11-22,593.659973,596.150024,593.150024,595.510010,582.077271,38226400,3.000000,640.284998,38.2264,1189.300049
2024-11-25,599.520020,600.859985,595.200012,597.530029,584.051575,42441400,5.659973,642.456732,42.4414,1196.059998
2024-11-26,598.799988,601.330017,598.070007,600.650024,587.101318,45621300,3.260010,645.811450,45.6213,1199.400024


#### 1.11.6 **Important: SettingWithCopyWarning**

You likely are getting setting with copy warnings. We need to address that now so that you understand what is causing it and how to avoid it, since you will likely encounter it OFTEN when coding.

The SettingWithCopyWarning appears when you attempt to modify a slice of a DataFrame, which may be a view rather than a new, independent copy. This warning is Pandas' way of alerting you that changes might not behave as expected. For example, modifying the slice may or may not affect the original DataFrame, leading to unpredictable results.

Essentially, Pandas can get confused when doing these kinds of operations. This is why, it often better to create a copy of a dataframe using `df2 = df1.copy()` and then working with the copy.

So, always check whether you are working with a copy or a slice of the original DataFrame. Using .copy() or .loc[] ensures clarity and avoids unexpected behavior. **This is critical for maintaining data integrity**, especially when modifying filtered subsets of your data.

Here are the "best practices" to deal with this:

1. Use `.copy()`: When creating a new DataFrame from a filtered slice, always use `.copy()` to ensure you're working with an independent copy.

 - Example: `df_spy_copy = df_spy[df_spy['Adj Close'] > 150].copy()`

2. Use `.loc[]` for Assignment: When modifying values in a DataFrame, use `.loc[]` to explicitly specify the rows and columns you want to update.

 - Example: `df_spy.loc[df_spy['Adj Close'] > 150, 'Volume (M)'] = df_spy['Volume'] / 1_000_000`

3. Understand the Warning: If you're certain your changes won't affect the original DataFrame, the warning can usually be ignored. However, using `.copy()` is safer and avoids ambiguity.

#### 1.11.7 The Proof of the Pudding is in the Tasting
We said that vectorization is fast. And now we will prove it, and you will also learn how to time operations in Python.

Remember that we said the alternative here is using for loops? Let's try that out and time how long it takes.

We will use the example of adding 10% to all Adj Close prices in df_spy

In [71]:
import time

# Vectorized approach
start_vec = time.time()
df_spy['Adj Close + 10%'] = df_spy['Adj Close'] * 1.10
end_vec = time.time()

# Print timing results
print("Vectorized approach time:", end_vec - start_vec)

Vectorized approach time: 0.0004470348358154297


In [72]:
# Loop-based approach
start_loop = time.time()
adj_close_10pct = []
for price in df_spy['Adj Close']:
    adj_close_10pct.append(price * 1.10)
df_spy['Adj Close + 10% (Loop)'] = adj_close_10pct
end_loop = time.time()

print("Loop-based approach time:", end_loop - start_loop)

# Comparing speeds: Vectorized vs Loop
percent_diff = (end_loop - start_loop) / (end_vec - start_vec)
print(f"The vectorized approach is {np.round(percent_diff, 2)} times faster than the loop!")


Loop-based approach time: 0.0017633438110351562
The vectorized approach is 3.94 times faster than the loop!


### 1.12 Index Sorting and Operations in Pandas

Efficiently managing the index of a DataFrame and sorting data are essential for organizing and analyzing large datasets. The methods `.reset_index()`, `.set_index()`, `.sort_index()`, and sorting by a column allow you to structure and retrieve data in meaningful ways.


#### 1.12.1 `reset_index()`

**What It Does**

reset_index() moves the current index (e.g., DatetimeIndex) to a regular column and resets the index to the default integer-based one.

In [73]:
# Reset index to default integers and keep the original index as a column
df_reset = df_spy.reset_index()
print(df_reset.head().round(2))


Price       Date  Adj Close   Close    High     Low    Open    Volume  \
0     2005-01-03      80.97  120.30  121.76  119.90  121.56  55748000   
1     2005-01-04      79.98  118.83  120.54  118.44  120.46  69167600   
2     2005-01-05      79.43  118.01  119.25  118.00  118.74  65667300   
3     2005-01-06      79.84  118.61  119.15  118.26  118.44  47814700   
4     2005-01-07      79.72  118.44  119.23  118.13  118.97  55847700   

Price  Daily % Change  Adj Close + 10%  High - Low  Volume (M)  \
0                 NaN            89.07        1.86       55.75   
1               -0.01            87.98        2.10       69.17   
2               -0.01            87.38        1.25       65.67   
3                0.01            87.82        0.89       47.81   
4               -0.00            87.69        1.10       55.85   

Price  Log(Adj Close)  Abs(Daily % Change) Above 150  Adj Close + 10% (Loop)  
0                4.39                  NaN        No                   89.07  
1     

You can see that the datetime index 'Date' is now a column in the dataframe and the index is now ordered integers.

There is also the option to drop the 'Date' column so that we reset the index to integers and then remove the Date column from the resulting dataframe:

In [74]:
df_reset = df_spy.reset_index(drop=True)  # note that drop=False is the default
print(df_reset.head().round(2))

Price  Adj Close   Close    High     Low    Open    Volume  Daily % Change  \
0          80.97  120.30  121.76  119.90  121.56  55748000             NaN   
1          79.98  118.83  120.54  118.44  120.46  69167600           -0.01   
2          79.43  118.01  119.25  118.00  118.74  65667300           -0.01   
3          79.84  118.61  119.15  118.26  118.44  47814700            0.01   
4          79.72  118.44  119.23  118.13  118.97  55847700           -0.00   

Price  Adj Close + 10%  High - Low  Volume (M)  Log(Adj Close)  \
0                89.07        1.86       55.75            4.39   
1                87.98        2.10       69.17            4.38   
2                87.38        1.25       65.67            4.37   
3                87.82        0.89       47.81            4.38   
4                87.69        1.10       55.85            4.38   

Price  Abs(Daily % Change) Above 150  Adj Close + 10% (Loop)  
0                      NaN        No                   89.07  
1       

**Why It’s Important**

- Flattening the DataFrame: Makes the DataFrame easier to work with when the index isn’t needed for operations like merging or exporting.
- Flexibility: Enables you to manipulate or reassign the index without losing information.

#### 1.12.2 `set_index()`

**What It Does**

`set_index()` assigns one or more columns as the new index of the DataFrame.



In [75]:
# Set the 'Date' column as the new index
df_set = df_spy.reset_index().set_index('Date')
print(df_set.head().round(2))


Price       Adj Close   Close    High     Low    Open    Volume  \
Date                                                              
2005-01-03      80.97  120.30  121.76  119.90  121.56  55748000   
2005-01-04      79.98  118.83  120.54  118.44  120.46  69167600   
2005-01-05      79.43  118.01  119.25  118.00  118.74  65667300   
2005-01-06      79.84  118.61  119.15  118.26  118.44  47814700   
2005-01-07      79.72  118.44  119.23  118.13  118.97  55847700   

Price       Daily % Change  Adj Close + 10%  High - Low  Volume (M)  \
Date                                                                  
2005-01-03             NaN            89.07        1.86       55.75   
2005-01-04           -0.01            87.98        2.10       69.17   
2005-01-05           -0.01            87.38        1.25       65.67   
2005-01-06            0.01            87.82        0.89       47.81   
2005-01-07           -0.00            87.69        1.10       55.85   

Price       Log(Adj Close)  Abs(

ok, the above example might seem a little odd. We reset the index and made 'Date' a column, and then we assigned the 'Date' column as the index, thus restoring the orignial structure.

Notice that we also created a new dataframe by using `df_set =`

But what if we just wanted to work with the original one and not create a new one.

We could have done this:

In [76]:
df_spy = df_spy.reset_index()
df_spy = df_spy.set_index('Date')
print(df_spy)

# We are essentially creating a copy of df_spy and calling it df_spy???
# That's not efficient.

Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2005-01-03   80.973534  120.300003  121.760002  119.900002  121.559998   
2005-01-04   79.984070  118.830002  120.540001  118.440002  120.459999   
2005-01-05   79.432137  118.010002  119.250000  118.000000  118.739998   
2005-01-06   79.836021  118.610001  119.150002  118.260002  118.440002   
2005-01-07   79.721573  118.440002  119.230003  118.129997  118.970001   
...                ...         ...         ...         ...         ...   
2024-11-22  582.077271  595.510010  596.150024  593.150024  593.659973   
2024-11-25  584.051575  597.530029  600.859985  595.200012  599.520020   
2024-11-26  587.101318  600.650024  601.330017  598.070007  598.799988   
2024-11-27  585.322327  598.830017  600.849976  597.280029  600.460022   
2024-11-29  588.958374  602.549988  603.349976  599.380005  599.659973   

Price         Volume  Daily % Change 

But that could also be done using `inplace=True` so that we don't create a copy and work directly on the original.

In [77]:
df_spy.reset_index(drop=False, inplace=True)
print(df_spy.head().round(2))

# In the output you will see we accomplished the exact same thing

Price       Date  Adj Close   Close    High     Low    Open    Volume  \
0     2005-01-03      80.97  120.30  121.76  119.90  121.56  55748000   
1     2005-01-04      79.98  118.83  120.54  118.44  120.46  69167600   
2     2005-01-05      79.43  118.01  119.25  118.00  118.74  65667300   
3     2005-01-06      79.84  118.61  119.15  118.26  118.44  47814700   
4     2005-01-07      79.72  118.44  119.23  118.13  118.97  55847700   

Price  Daily % Change  Adj Close + 10%  High - Low  Volume (M)  \
0                 NaN            89.07        1.86       55.75   
1               -0.01            87.98        2.10       69.17   
2               -0.01            87.38        1.25       65.67   
3                0.01            87.82        0.89       47.81   
4               -0.00            87.69        1.10       55.85   

Price  Log(Adj Close)  Abs(Daily % Change) Above 150  Adj Close + 10% (Loop)  
0                4.39                  NaN        No                   89.07  
1     

And now to set the index using inplace:

In [78]:
df_spy.set_index('Date', inplace=True)
print(df_spy.head().round(2))

Price       Adj Close   Close    High     Low    Open    Volume  \
Date                                                              
2005-01-03      80.97  120.30  121.76  119.90  121.56  55748000   
2005-01-04      79.98  118.83  120.54  118.44  120.46  69167600   
2005-01-05      79.43  118.01  119.25  118.00  118.74  65667300   
2005-01-06      79.84  118.61  119.15  118.26  118.44  47814700   
2005-01-07      79.72  118.44  119.23  118.13  118.97  55847700   

Price       Daily % Change  Adj Close + 10%  High - Low  Volume (M)  \
Date                                                                  
2005-01-03             NaN            89.07        1.86       55.75   
2005-01-04           -0.01            87.98        2.10       69.17   
2005-01-05           -0.01            87.38        1.25       65.67   
2005-01-06            0.01            87.82        0.89       47.81   
2005-01-07           -0.00            87.69        1.10       55.85   

Price       Log(Adj Close)  Abs(

#### 1.12.3 `sort_index()`

**What It Does**

`sort_index()` reorders the rows (or columns) in ascending or descending order based on the index.

In [79]:
# Sort the index in ascending order
df_sorted_index = df_spy.sort_index()

# Sort the index in descending order
df_sorted_desc = df_spy.sort_index(ascending=False)

print(df_sorted_desc.head().round(2))

# Notice the dataframe is now ordered by date in descending order

Price       Adj Close   Close    High     Low    Open    Volume  \
Date                                                              
2024-11-29     588.96  602.55  603.35  599.38  599.66  30177400   
2024-11-27     585.32  598.83  600.85  597.28  600.46  34000200   
2024-11-26     587.10  600.65  601.33  598.07  598.80  45621300   
2024-11-25     584.05  597.53  600.86  595.20  599.52  42441400   
2024-11-22     582.08  595.51  596.15  593.15  593.66  38226400   

Price       Daily % Change  Adj Close + 10%  High - Low  Volume (M)  \
Date                                                                  
2024-11-29            0.01           647.85        3.97       30.18   
2024-11-27           -0.00           643.85        3.57       34.00   
2024-11-26            0.01           645.81        3.26       45.62   
2024-11-25            0.00           642.46        5.66       42.44   
2024-11-22            0.00           640.28        3.00       38.23   

Price       Log(Adj Close)  Abs(

**Key Options:**

- axis=0: Sort rows (default).
- axis=1: Sort columns.
- inplace=True: Modify the original DataFrame.

**Why It’s Important**
- Consistency: Ensures the DataFrame is ordered correctly by its index, critical for time-series data.
- Readability: Makes the data easier to interpret when the index is in order.

#### 1.12.4 Sorting by a Column

**What It Does**

Sort rows based on the values in one or more columns.

In [80]:
# Sort by 'Adj Close' in ascending order
df_sorted = df_spy.sort_values(by='Adj Close')
print(df_sorted.head().round(2))

Price       Adj Close  Close   High    Low   Open     Volume  Daily % Change  \
Date                                                                           
2009-03-09      49.56  68.11  70.00  67.73  67.95  379905300           -0.01   
2009-03-05      50.06  68.80  71.73  68.17  70.10  485549400           -0.04   
2009-03-06      50.15  68.92  70.45  67.10  69.40  490470000            0.00   
2009-03-03      50.98  70.07  71.70  69.64  71.61  443761000           -0.01   
2009-03-02      51.37  70.60  73.92  70.37  72.52  426452600           -0.05   

Price       Adj Close + 10%  High - Low  Volume (M)  Log(Adj Close)  \
Date                                                                  
2009-03-09            54.51        2.27      379.91            3.90   
2009-03-05            55.07        3.56      485.55            3.91   
2009-03-06            55.16        3.35      490.47            3.91   
2009-03-03            56.08        2.06      443.76            3.93   
2009-03-02   

In [81]:
# Sort by 'Adj Close' in descending order
df_sorted_desc = df_spy.sort_values(by='Adj Close', ascending=False)
print(df_sorted_desc.head())


Price        Adj Close       Close        High         Low        Open  \
Date                                                                     
2024-11-29  588.958374  602.549988  603.349976  599.380005  599.659973   
2024-11-26  587.101318  600.650024  601.330017  598.070007  598.799988   
2024-11-27  585.322327  598.830017  600.849976  597.280029  600.460022   
2024-11-11  585.253845  598.760010  600.169983  597.000000  599.809998   
2024-11-08  584.696655  598.190002  599.640015  596.169983  596.169983   

Price         Volume  Daily % Change  Adj Close + 10%  High - Low  Volume (M)  \
Date                                                                            
2024-11-29  30177400        0.006212       647.854211    3.969971     30.1774   
2024-11-26  45621300        0.005222       645.811450    3.260010     45.6213   
2024-11-27  34000200       -0.003030       643.854559    3.569946     34.0002   
2024-11-11  37586800        0.000953       643.779230    3.169983     37.586

In [82]:
# Sort by multiple columns: 'Volume' (descending), then 'Adj Close' (ascending)
df_sorted_multi = df_spy.sort_values(by=['Volume', 'Adj Close'], ascending=[False, True])
print(df_sorted_multi.head())

Price       Adj Close       Close        High         Low        Open  \
Date                                                                    
2008-10-10  63.874958   88.500000   93.940002   83.580002   86.760002   
2008-11-20  54.456097   75.449997   82.510002   75.050003   80.129997   
2008-09-18  86.161903  120.070000  121.790001  113.800003  118.050003   
2008-11-13  65.802032   91.169998   91.730003   82.089996   86.129997   
2008-10-08  70.377945   97.510002  102.180000   96.809998   97.519997   

Price          Volume  Daily % Change  Adj Close + 10%  High - Low  \
Date                                                                 
2008-10-10  871026300       -0.024255        70.262454   10.360001   
2008-11-20  814180400       -0.074233        59.901706    7.459999   
2008-09-18  776114700        0.029671        94.778094    7.989998   
2008-11-13  753141900        0.062340        72.382236    9.640007   
2008-10-08  725414800       -0.025192        77.415739    5.370003  

**Why It’s Important**
- Prioritization: Helps identify rows based on ranking, such as the highest volume trading days or the lowest closing prices.
- Preparation: Essential for tasks like ranking, filtering top/bottom rows, or visualizing sorted data.

### 1.13 Dealing with NaN Values in Pandas

`NaN` values represent missing or undefined data, which is common in financial datasets due to market holidays, incomplete data feeds, or errors. Handling these values correctly is essential to maintain the accuracy and reliability of your analysis.

#### 1.13.1 Finding `NaN` Values

**What It Does**

Identifies where missing values (NaN) exist in a DataFrame or Series.

**How to Use It**

i. Check for Any Missing Values


In [83]:
print(df_spy.isnull())

# Returns a DataFrame of the same shape, where True indicates missing values.
# This isn't exactly helpful...

Price       Adj Close  Close   High    Low   Open  Volume  Daily % Change  \
Date                                                                        
2005-01-03      False  False  False  False  False   False            True   
2005-01-04      False  False  False  False  False   False           False   
2005-01-05      False  False  False  False  False   False           False   
2005-01-06      False  False  False  False  False   False           False   
2005-01-07      False  False  False  False  False   False           False   
...               ...    ...    ...    ...    ...     ...             ...   
2024-11-22      False  False  False  False  False   False           False   
2024-11-25      False  False  False  False  False   False           False   
2024-11-26      False  False  False  False  False   False           False   
2024-11-27      False  False  False  False  False   False           False   
2024-11-29      False  False  False  False  False   False           False   

ii. Check if any missing values exist

In [84]:
print(df_spy.isnull().any())
# This is a much more useful output!

Price
Adj Close                 False
Close                     False
High                      False
Low                       False
Open                      False
Volume                    False
Daily % Change             True
Adj Close + 10%           False
High - Low                False
Volume (M)                False
Log(Adj Close)            False
Abs(Daily % Change)        True
Above 150                 False
Adj Close + 10% (Loop)    False
dtype: bool


iii. Count missing values per column

In [85]:
print(df_spy.isnull().sum())


Price
Adj Close                 0
Close                     0
High                      0
Low                       0
Open                      0
Volume                    0
Daily % Change            1
Adj Close + 10%           0
High - Low                0
Volume (M)                0
Log(Adj Close)            0
Abs(Daily % Change)       1
Above 150                 0
Adj Close + 10% (Loop)    0
dtype: int64


This output makes perfect sense. We only have two missing values. This occurred when we did the percent change calculations, and the first row gives an NaN since there were no rows above on which to calculate the percent change.

**Why It’s Important**

Locates gaps in data, helping you decide how to address missing values.
Ensures that methods like mean() or pct_change() are not skewed by NaN values.

So now that we know how to detect NaN, what do we do about them?

Before we continue, let's create a sample dataframe with random data. This will make all subsequent examples clearer.


In [86]:
# Create a date range for the index
date_range = pd.date_range(start="2023-01-01", periods=100, freq="D")

# Generate random numeric data with NaN values
np.random.seed(42)
data = {
    "Price": np.random.choice([np.nan, 100, 105, 110], size=100),
    "Volume": np.random.choice([np.nan, 1000, 1200, 1500], size=100),
    "Returns": np.random.choice([np.nan, 0.01, -0.02, 0.03], size=100)
}

# Create a DataFrame
sample_df = pd.DataFrame(data, index=date_range)

sample_df.head()

,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-03,NaN,1000.0,-0.02
2023-01-04,105.0,1500.0,NaN
2023-01-05,105.0,1000.0,0.03


Feel free to add some cells here and explore the sample data. We will use `.info()` to show that there are missing values

In [87]:
sample_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2023-01-01 to 2023-04-10
Freq: D
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    80 non-null     float64
 1   Volume   74 non-null     float64
 2   Returns  72 non-null     float64
dtypes: float64(3)
memory usage: 3.1 KB


#### 1.13.2 `dropna()`

**What It Does**
- Removes rows or columns that contain missing values.

**i. Drop rows that have ANY missing values**

In [88]:
# This drops all rows that have at least one missing value
# we will create a new df for these examples and not use inplace=True

# 1) Default, axis=0 --> drop rows that have at least one NaN
df_dropped_axis0 = sample_df.dropna()
df_dropped_axis0.info()
df_dropped_axis0.head(15)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 40 entries, 2023-01-01 to 2023-04-08
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    40 non-null     float64
 1   Volume   40 non-null     float64
 2   Returns  40 non-null     float64
dtypes: float64(3)
memory usage: 1.2 KB


,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-05,105.0,1000.0,0.03
2023-01-09,105.0,1000.0,0.01
2023-01-11,105.0,1500.0,-0.02
2023-01-12,105.0,1200.0,-0.02
2023-01-14,105.0,1000.0,-0.02
2023-01-15,110.0,1200.0,-0.02
2023-01-19,110.0,1500.0,0.03
2023-01-21,100.0,1500.0,-0.02


We can see that there are many missing days now as they have been removed.

The `.info()` shows that we now have only 40 days, and that 60 were dropped!

In [89]:
# This drops all rows that have at least one missing value
# we will create a new df for these examples and not use inplace=True

# 2) axis=1 --> drop columns that have at least one NaN
df_dropped_axis1 = sample_df.dropna(axis=1)
df_dropped_axis1.info()
df_dropped_axis1.head(15)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2023-01-01 to 2023-04-10
Freq: D
Empty DataFrame


""
2023-01-01
2023-01-02
2023-01-03
2023-01-04
2023-01-05
2023-01-06
2023-01-07
2023-01-08
2023-01-09
2023-01-10


We got an empty dataframe! This should come as no surprise because all columns had at least one NaN

**ii. Dropping rows only if ALL values are NaN**

What if we didn't want to drop all rows if they had only one NaN? This isn't what we always want to do.

The next example shows how you can instruct Python to delete rows that have ALL NaNs in a row/column. This means that every value in the row must be NaN for it to be removed.

In this case, we add the parameter 'how': `dropna(how='all')`

In [90]:
df_drop_all = sample_df.dropna(how='all')
df_drop_all.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2023-01-01 to 2023-04-10
Freq: D
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    80 non-null     float64
 1   Volume   74 non-null     float64
 2   Returns  72 non-null     float64
dtypes: float64(3)
memory usage: 3.1 KB


In this case, no rows were dropped since there were no rows that contained NaN for each column/value.
We would expect the same result for:
`dropna(axis=1, how='all')`

Note: the default behaviour is: `dropna(how='any')` which is the same as `dropna()`

**iii. Dropping rows/columns only if there are a minimum amount of missing values**

In [91]:
# This will remove rows that have at least 2 NaN values
df_thresh = sample_df.dropna(thresh=2)
df_thresh.info()

# Compare against the original
sample_df.info()
# You will see that we removed a total of 14 rows

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 86 entries, 2023-01-01 to 2023-04-10
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    75 non-null     float64
 1   Volume   69 non-null     float64
 2   Returns  68 non-null     float64
dtypes: float64(3)
memory usage: 2.7 KB
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2023-01-01 to 2023-04-10
Freq: D
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    80 non-null     float64
 1   Volume   74 non-null     float64
 2   Returns  72 non-null     float64
dtypes: float64(3)
memory usage: 3.1 KB


#### 1.13.3 `fillna()`

We might not always want to delete the rows/columns with NaN values.
Very often, we need to do something with them so that we don't lose whole rows of valuable data.

We will look at the most popular ways that you can deal with missing values.

**i. Fill with a fixed value (example: fill all NaN with 0)**

In [92]:
df_filled_0 = sample_df.fillna(0)
df_filled_0.info()
df_filled_0.head()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 100 entries, 2023-01-01 to 2023-04-10
Freq: D
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Price    100 non-null    float64
 1   Volume   100 non-null    float64
 2   Returns  100 non-null    float64
dtypes: float64(3)
memory usage: 3.1 KB


,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-03,0.0,1000.0,-0.02
2023-01-04,105.0,1500.0,0.00
2023-01-05,105.0,1000.0,0.03


**ii. Forward fill with `ffill`**

A forward fill will take the most recent valid value and use that.
Look at the original, and then predict what the result will be once we do a forward fill:

In [93]:
sample_df.head()

,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-03,NaN,1000.0,-0.02
2023-01-04,105.0,1500.0,NaN
2023-01-05,105.0,1000.0,0.03


In [94]:
df_ffill = sample_df.ffill()
df_ffill.head()

,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-03,110.0,1000.0,-0.02
2023-01-04,105.0,1500.0,-0.02
2023-01-05,105.0,1000.0,0.03


**iii. Backward fill with `bfill`**


In [95]:
df_bfill = sample_df.bfill()
df_bfill.head()

,Price,Volume,Returns
2023-01-01,105.0,1200.0,-0.02
2023-01-02,110.0,1000.0,0.03
2023-01-03,105.0,1000.0,-0.02
2023-01-04,105.0,1500.0,0.03
2023-01-05,105.0,1000.0,0.03


**iv. Filling NaN with mean or median values**

This is a very common method: fill all missing NaN with the mean values found in the column. Since we have a central tendency theorem, it makes sense to assign the mean to the missing values. This way we don;t change the overall distribution or descriptive statistics.

Of course some data sets could be skewed, and it would be more appropriate to use the median.




In [96]:
# Let's examine the mean values first
print(sample_df.mean())

df_fill_mean = sample_df.fillna(sample_df.mean())
df_fill_mean.head()

Price       105.250000
Volume     1243.243243
Returns       0.007917
dtype: float64


,Price,Volume,Returns
2023-01-01,105.00,1200.0,-0.020000
2023-01-02,110.00,1000.0,0.030000
2023-01-03,105.25,1000.0,-0.020000
2023-01-04,105.00,1500.0,0.007917
2023-01-05,105.00,1000.0,0.030000


Add some cells and try to do this with median!

With time series, you would really have to think about the consequences
of using mean/median.

Let's take a specific situation:

Imagine you download daily pricing data for an ETF. You notice that some days have missing values for price. This could happen if there were no trades in  a given day so the data source, when detecting zero volume, might just assign an NaN to the price for that day.

What do you think is the most logical way to deal with this?

A forward fill would make sense where you just take the previous closing value and assign it to the NaN.


---



### 1.14 Time-Series Operations

Now we get to some of the most powerful methods in Pandas for financial analysis. These are the methods that will allow you to build trading signals, create indicators, and transform your data across time.

Everything we've done so far has been about understanding the structure of our data and basic calculations. But trading strategies require us to look at data *over time*: What was the average price over the last 20 days? What was yesterday's close? What is the cumulative return since we started? How do I convert daily prices to monthly returns?

This section will cover the essential time-series methods you need to master:

1. `.rolling()` — Moving window calculations (e.g., 20-day moving average)
2. `.shift()` — Accessing previous or future values (lagging and leading)
3. `.cumsum()` and `.cumprod()` — Cumulative operations (essential for cumulative returns)
4. `.resample()` — Changing the frequency of your data (e.g., daily to monthly)

These methods are the building blocks of almost every trading strategy you will encounter. Let's dive in.



#### 1.14.1 `.rolling()` — Moving Window Calculations

**What It Does:** The `.rolling()` method creates a "sliding window" over your data, allowing you to perform calculations across a specified number of rows. Think of it as a window that moves down your DataFrame, one row at a time, performing a calculation at each step.

**Why It's Important:** Moving averages are the foundation of countless trading strategies. When you hear someone say "the 50-day moving average crossed above the 200-day moving average," they are talking about `.rolling()`. Beyond moving averages, you can calculate rolling standard deviations (for volatility), rolling sums, rolling min/max, and much more.

**When to Use It:**
- Calculating moving averages for trend-following strategies
- Measuring rolling volatility (standard deviation of returns)
- Smoothing noisy data to identify underlying trends
- Building indicators like Bollinger Bands (which use rolling mean and rolling standard deviation)

The syntax is straightforward: you specify the window size (number of periods), then chain the aggregation method you want.

`.rolling(window=20).mean()` — 20-period moving average

`.rolling(window=20).std()` — 20-period rolling standard deviation

Let's see it in action with our SPY data.

In [97]:
# First, let's recreate df_spy since we've really addedd a lot to it
df_spy = yf.download(tickers=ticker, start=start_date, end=end_date, auto_adjust=False)
print(df_spy.info())

# Remove the Multi Index
df_spy.columns = df_spy.columns.get_level_values(0)
print(df_spy.info())

[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5012 entries, 2005-01-03 to 2024-11-29
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   (Adj Close, SPY)  5012 non-null   float64
 1   (Close, SPY)      5012 non-null   float64
 2   (High, SPY)       5012 non-null   float64
 3   (Low, SPY)        5012 non-null   float64
 4   (Open, SPY)       5012 non-null   float64
 5   (Volume, SPY)     5012 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 274.1 KB
None
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5012 entries, 2005-01-03 to 2024-11-29
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  5012 non-null   float64
 1   Close      5012 non-null   float64
 2   High       5012 non-null   float64
 3   Low        5012 non-null   float64
 4   Open       5012 non-null   float64
 5   Volume     5012 non-null   int64  

In [98]:
# Calculate the 20-day moving average of the Adjusted Close price
df_spy['MA_20'] = df_spy['Adj Close'].rolling(window=20).mean()

# Calculate the 50-day moving average
df_spy['MA_50'] = df_spy['Adj Close'].rolling(window=50).mean()

# Let's see the results
print(df_spy[['Adj Close', 'MA_20', 'MA_50']].tail(10).round(2))

Price       Adj Close   MA_20   MA_50
Date                                 
2024-11-15     572.54  571.26  562.32
2024-11-18     574.88  571.48  563.17
2024-11-19     576.98  571.82  564.01
2024-11-20     577.18  572.43  564.75
2024-11-21     580.28  573.14  565.46
2024-11-22     582.08  573.94  566.15
2024-11-25     584.05  574.76  566.86
2024-11-26     587.10  575.68  567.63
2024-11-27     585.32  576.60  568.39
2024-11-29     588.96  578.26  569.04


Notice a few things from the output:

1. The moving averages are smoother than the raw Adjusted Close prices. That is the whole point. They filter out the daily noise.

2. If you scroll to the top of your DataFrame (using `.head(50)`), you will see that `MA_20` has NaN values for the first 19 rows, and `MA_50` has NaN values for the first 49 rows. Why? Because Pandas cannot calculate a 20-day average until it has 20 days of data!

3. The longer the window, the smoother (and slower to react) the moving average becomes. This is the classic trade-off between responsiveness and noise reduction.

Let's also calculate rolling volatility, which is the 20-day rolling standard deviation of daily returns. This is how professionals measure risk over time.

In [99]:
# First, let's make sure we have daily returns
df_spy['Daily_Return'] = df_spy['Adj Close'].pct_change()

# Calculate 20-day rolling volatility (standard deviation of returns)
df_spy['Volatility_20'] = df_spy['Daily_Return'].rolling(window=20).std()

# Let's see the last 10 rows
print(df_spy[['Daily_Return', 'Volatility_20']].tail(10))

Price       Daily_Return  Volatility_20
Date                                   
2024-11-15     -0.012809       0.009114
2024-11-18      0.004097       0.009145
2024-11-19      0.003656       0.009170
2024-11-20      0.000339       0.008879
2024-11-21      0.005368       0.008928
2024-11-22      0.003099       0.008928
2024-11-25      0.003392       0.008931
2024-11-26      0.005222       0.008971
2024-11-27     -0.003030       0.008971
2024-11-29      0.006212       0.007489


#### 1.14.2 `.shift()` — Lagging and Leading Data

**What It Does:** The `.shift()` method moves your data up or down by a specified number of rows. A positive number shifts the data down (lagging), and a negative number shifts the data up (leading).

**Why It's Important:** In trading, you constantly need to compare today's value to yesterday's value, or align signals with future returns. The `.shift()` method is how you do this. It is also critical for avoiding one of the most dangerous mistakes in backtesting: lookahead bias. If you accidentally use tomorrow's data to make today's decision, your backtest is worthless.

**When to Use It:**
- Calculating returns manually: `(price - price.shift(1)

In [100]:
# Shift the Adjusted Close down by 1 row (yesterday's price)
df_spy['Prev_Close'] = df_spy['Adj Close'].shift(1)

# Now let's see today's close vs yesterday's close
print(df_spy[['Adj Close', 'Prev_Close']].head(10))

Price       Adj Close  Prev_Close
Date                             
2005-01-03  80.973564         NaN
2005-01-04  79.984070   80.973564
2005-01-05  79.432121   79.984070
2005-01-06  79.835976   79.432121
2005-01-07  79.721542   79.835976
2005-01-10  80.098503   79.721542
2005-01-11  79.546570   80.098503
2005-01-12  79.809090   79.546570
2005-01-13  79.169640   79.809090
2005-01-14  79.586952   79.169640


Take a look at the first row. The `Prev_Close` column is NaN because there is no row above it to pull from. This is exactly what you should expect.

Now look at the second row. The `Prev_Close` value matches the `Adj Close` from the first row. The data has been shifted down by one position.

This is incredibly useful. For example, you can now calculate daily returns manually:

In [101]:
# Calculate daily return manually using shift
df_spy['Manual_Return'] = (df_spy['Adj Close'] - df_spy['Prev_Close']) / df_spy['Prev_Close']

# Compare it to pct_change
print(df_spy[['Daily_Return', 'Manual_Return']].head(10))

Price       Daily_Return  Manual_Return
Date                                   
2005-01-03           NaN            NaN
2005-01-04     -0.012220      -0.012220
2005-01-05     -0.006901      -0.006901
2005-01-06      0.005084       0.005084
2005-01-07     -0.001433      -0.001433
2005-01-10      0.004728       0.004728
2005-01-11     -0.006891      -0.006891
2005-01-12      0.003300       0.003300
2005-01-13     -0.008012      -0.008012
2005-01-14      0.005271       0.005271


The values are identical. The `.pct_change()` method is simply doing this calculation for you under the hood.

You can also shift in the opposite direction using negative numbers. This pulls future data into the current row. While this is useful for analysis (for example, comparing today's signal to tomorrow's return), you must be extremely careful never to use this in a live trading system or backtest. Using future data to make past decisions is called lookahead bias, and it will make your backtest look amazing while your live strategy loses money.

In [102]:
# Shift the Adjusted Close UP by 1 row (tomorrow's price)
df_spy['Next_Close'] = df_spy['Adj Close'].shift(-1)

# See the result
print(df_spy[['Adj Close', 'Next_Close']].tail(10))

Price        Adj Close  Next_Close
Date                              
2024-11-15  572.537292  574.883179
2024-11-18  574.883179  576.984680
2024-11-19  576.984680  577.180237
2024-11-20  577.180237  580.278687
2024-11-21  580.278687  582.077209
2024-11-22  582.077209  584.051636
2024-11-25  584.051636  587.101379
2024-11-26  587.101379  585.322327
2024-11-27  585.322327  588.958496
2024-11-29  588.958496         NaN


Notice that the last row now has NaN in the `Next_Close` column. There is no future data to pull from.

The `.shift()` method is deceptively simple but absolutely essential. You will use it constantly when building trading strategies.


#### 1.14.3 `.cumsum()` and `.cumprod()` — Cumulative Operations

**What They Do:** These methods calculate running totals across your data. The `.cumsum()` method calculates the cumulative sum, while `.cumprod()` calculates the cumulative product.

**Why They're Important:** In finance, cumulative operations are essential for tracking the growth of an investment over time. If you want to know how much $1 invested at the start of your data would be worth today, you need `.cumprod()`. If you are working with log returns and want cumulative performance, you need `.cumsum()`.

**When to Use Them:**
- `.cumprod()` is used to calculate cumulative returns from simple returns. This is the standard way to build an equity curve.
- `.cumsum()` is used to calculate cumulative returns from log returns, or to track cumulative profit and loss in points or dollars.
- Both are used extensively in performance analysis and visualization.

The most common pattern you will see in quantitative finance is:

`(1 + returns).cumprod() - 1`

This converts a series of daily percentage returns into cumulative total return. Let's break this down step by step.

In [103]:
# Start with our daily returns (already calculated as Daily_Return)
# Step 1: Add 1 to each return to get the growth factor
# If today's return is 0.01 (1%), then the growth factor is 1.01

df_spy['Growth_Factor'] = 1 + df_spy['Daily_Return']

# Step 2: Calculate the cumulative product of growth factors
# This tells us the total growth multiplier from the start

df_spy['Cumulative_Growth'] = df_spy['Growth_Factor'].cumprod()

# Step 3: Subtract 1 to convert back to a return
# This is the total cumulative return from the first date

df_spy['Cumulative_Return'] = df_spy['Cumulative_Growth'] - 1

# Let's see the last 10 rows
print(df_spy[['Daily_Return', 'Growth_Factor', 'Cumulative_Growth',
              'Cumulative_Return']].tail(10))

Price       Daily_Return  Growth_Factor  Cumulative_Growth  Cumulative_Return
Date                                                                         
2024-11-15     -0.012809       0.987191           7.070669           6.070669
2024-11-18      0.004097       1.004097           7.099640           6.099640
2024-11-19      0.003656       1.003656           7.125593           6.125593
2024-11-20      0.000339       1.000339           7.128008           6.128008
2024-11-21      0.005368       1.005368           7.166273           6.166273
2024-11-22      0.003099       1.003099           7.188484           6.188484
2024-11-25      0.003392       1.003392           7.212868           6.212868
2024-11-26      0.005222       1.005222           7.250532           6.250532
2024-11-27     -0.003030       0.996970           7.228561           6.228561
2024-11-29      0.006212       1.006212           7.273466           6.273466


Let's walk through what each column is telling us:

1. `Daily_Return` is the percentage change from the previous day. A value of 0.01 means the price went up 1%.

2. `Growth_Factor` is simply 1 plus the daily return. If the price went up 1%, the growth factor is 1.01. If the price went down 2%, the growth factor is 0.98.

3. `Cumulative_Growth` is the cumulative product of all the growth factors from the beginning of our data. If this value is 4.5, it means $1 invested on day one is now worth $4.50.

4. `Cumulative_Return` is the cumulative growth minus 1, expressed as a return. A value of 3.5 means a 350% total return.

In practice, you will often see this written in a single line:

`(1 + df['returns']).cumprod() - 1`

This is the standard formula for building an equity curve, and you will use it in almost every backtest.

In [104]:
# The one-liner version (this is how you will typically see it)
df_spy['Cumulative_Return_v2'] = (1 + df_spy['Daily_Return']).cumprod() - 1

# Verify it matches our step-by-step calculation
print(df_spy[['Cumulative_Return', 'Cumulative_Return_v2']].tail(10))

Price       Cumulative_Return  Cumulative_Return_v2
Date                                               
2024-11-15           6.070669              6.070669
2024-11-18           6.099640              6.099640
2024-11-19           6.125593              6.125593
2024-11-20           6.128008              6.128008
2024-11-21           6.166273              6.166273
2024-11-22           6.188484              6.188484
2024-11-25           6.212868              6.212868
2024-11-26           6.250532              6.250532
2024-11-27           6.228561              6.228561
2024-11-29           6.273466              6.273466


Now let's look at `.cumsum()`. While `.cumprod()` is used for compounding returns, `.cumsum()` is used for additive accumulation.

**When to Use `.cumsum()`:**
- When working with log returns (which are additive, not multiplicative)
- When tracking cumulative profit and loss in dollar or point terms
- When you want a running total of any value

Let's see a simple example with cumulative sum of daily returns. Note that this is not the correct way to calculate cumulative performance from simple returns, but it is useful to see how `.cumsum()` works.

In [105]:
# Calculate cumulative sum of daily returns
df_spy['Cumsum_Return'] = df_spy['Daily_Return'].cumsum()

# Compare cumsum vs cumprod approaches
print(df_spy[['Daily_Return', 'Cumsum_Return', 'Cumulative_Return']].tail(10))

Price       Daily_Return  Cumsum_Return  Cumulative_Return
Date                                                      
2024-11-15     -0.012809       2.317612           6.070669
2024-11-18      0.004097       2.321709           6.099640
2024-11-19      0.003656       2.325365           6.125593
2024-11-20      0.000339       2.325704           6.128008
2024-11-21      0.005368       2.331072           6.166273
2024-11-22      0.003099       2.334172           6.188484
2024-11-25      0.003392       2.337564           6.212868
2024-11-26      0.005222       2.342785           6.250532
2024-11-27     -0.003030       2.339755           6.228561
2024-11-29      0.006212       2.345967           6.273466


Notice that `Cumsum_Return` and `Cumulative_Return` are different values. This is expected.

The `.cumsum()` approach simply adds up all the daily returns. This ignores the compounding effect of returns. If you made 10% on day one, and then 10% on day two, `.cumsum()` would say you made 20%. But in reality, you made 10% on your original amount, and then 10% on your new (larger) amount. The true return is 21% (1.10 x 1.10 = 1.21).

For simple percentage returns, always use `.cumprod()` to calculate cumulative performance. The `.cumsum()` method is appropriate when working with log returns or when you need a simple running total.


#### 1.14.4 `.resample()` — Changing Data Frequency

**What It Does:** The `.resample()` method allows you to change the frequency of your time-series data. You can convert daily data to weekly, weekly to monthly, or even go the other direction if you have higher-frequency data (though upsampling requires you to decide how to fill in the gaps).

**Why It's Important:** Different trading strategies operate on different time horizons. A long-term investor might only care about monthly returns, while a day trader needs minute-by-minute data. The ability to change frequency is essential for multi-timeframe analysis and for aligning datasets that come in different frequencies.

**When to Use It:**
- Converting daily prices to monthly prices for long-term analysis
- Aggregating tick or minute data into daily bars
- Aligning data from different sources that have different frequencies
- Calculating monthly or weekly returns from daily data

The syntax requires two parts: the frequency code and an aggregation method.

`.resample('ME')` — Resample to month-end frequency

`.resample('W')` — Resample to weekly frequency

`.resample('YE')` — Resample to year-end frequency

After specifying the frequency, you must tell Pandas how to aggregate the data within each period. This is where many beginners make mistakes.

In [106]:
# Resample daily data to monthly frequency using the last value of each month
df_spy_monthly = df_spy['Adj Close'].resample('ME').last()

# Let's see the result
print(df_spy_monthly.tail(12))

Date
2023-12-31    460.248444
2024-01-31    467.578583
2024-02-29    491.980011
2024-03-31    508.068726
2024-04-30    487.583557
2024-05-31    512.245422
2024-06-30    530.317566
2024-07-31    536.739197
2024-08-31    549.280457
2024-09-30    560.817871
2024-10-31    555.813232
2024-11-30    588.958496
Freq: ME, Name: Adj Close, dtype: float64


We now have month-end prices for SPY. Each row represents the last trading day of that month, and the value is the closing price on that day.

But why did we use `.last()`? This is critical to understand.

We now have month-end prices for SPY. Each row represents the last trading day of that month, and the value is the closing price on that day.

But why did we use `.last()`? This is critical to understand.

#### 1.14.5. The Resampling Lab: Choosing the Right Aggregation Method

When you resample data, you are taking many rows and collapsing them into one. But how should Pandas combine those values? The answer depends entirely on what the data represents.

**The Golden Rules:**

1. **For prices, use `.last()`** — You want the closing price at the end of the period. The last price of the month is the month-end price. Using `.mean()` would give you the average price during the month, which is not what you typically want for calculating returns.

2. **For volume, use `.sum()`** — Volume is cumulative. If you traded 1 million shares on Monday and 2 million on Tuesday, the weekly volume is 3 million. Using `.last()` would only give you Tuesday's volume and throw away Monday's data.

3. **For returns, be careful** — If you have daily returns and want monthly returns, you cannot simply use `.sum()` or `.mean()`. You must compound them using the formula we learned: `(1 + daily_returns).resample('ME').prod() - 1`

Let's see what happens when you use the wrong aggregation method.

In [107]:
# Create a DataFrame with both price and volume for our demo
df_resample_demo = df_spy[['Adj Close', 'Volume']].copy()

# WRONG: Using mean for prices
monthly_price_mean = df_resample_demo['Adj Close'].resample('ME').mean()

# CORRECT: Using last for prices
monthly_price_last = df_resample_demo['Adj Close'].resample('ME').last()

# Compare the two
print("Last 6 months comparison - Price aggregation:")
print(f"\n{'Month':<12} {'Using mean()':<15} {'Using last()':<15}")
print("-" * 42)

for date, mean_val, last_val in zip(monthly_price_mean.tail(6).index,
                                      monthly_price_mean.tail(6).values,
                                      monthly_price_last.tail(6).values):
    print(f"{str(date.date()):<12} {mean_val:<15.2f} {last_val:<15.2f}")

Last 6 months comparison - Price aggregation:

Month        Using mean()    Using last()   
------------------------------------------
2024-06-30   525.54          530.32         
2024-07-31   537.92          536.74         
2024-08-31   532.59          549.28         
2024-09-30   547.21          560.82         
2024-10-31   564.36          555.81         
2024-11-30   578.26          588.96         


The values are different. The `.mean()` column shows the average price throughout each month, while `.last()` shows the actual closing price on the last day of the month.

If you were to calculate monthly returns using the mean prices, your returns would be wrong. Always use `.last()` for prices when calculating returns.

Now let's look at volume.

In [108]:
# WRONG: Using last for volume (only captures final day's volume)
monthly_volume_last = df_resample_demo['Volume'].resample('ME').last()

# CORRECT: Using sum for volume (captures total volume for the month)
monthly_volume_sum = df_resample_demo['Volume'].resample('ME').sum()

# Compare the two
print("Last 6 months comparison - Volume aggregation:")
print(f"\n{'Month':<12} {'Using last()':<20} {'Using sum()':<20}")
print("-" * 52)

for date, last_val, sum_val in zip(monthly_volume_last.tail(6).index,
                                    monthly_volume_last.tail(6).values,
                                    monthly_volume_sum.tail(6).values):
    print(f"{str(date.date()):<12} {last_val:<20,.0f} {sum_val:<20,.0f}")

Last 6 months comparison - Volume aggregation:

Month        Using last()         Using sum()         
----------------------------------------------------
2024-06-30   76,144,500           888,367,600         
2024-07-31   65,663,400           1,038,465,500       
2024-08-31   62,700,100           1,244,599,000       
2024-09-30   63,557,400           1,044,988,300       
2024-10-31   60,182,500           976,068,800         
2024-11-30   30,177,400           901,760,600         


Look at the difference. Using `.last()` on volume gives you only the volume from the final trading day of the month. Using `.sum()` gives you the total volume traded during the entire month. The sum values are roughly 20 times larger because there are roughly 20 trading days in a month.

If you used `.last()` for volume analysis, you would be throwing away most of your data.

In [109]:
# The correct way to calculate monthly returns from daily prices
# Step 1: Get month-end prices using .last()
monthly_prices = df_spy['Adj Close'].resample('ME').last()

# Step 2: Calculate monthly returns using pct_change
monthly_returns = monthly_prices.pct_change()

# Let's see the result
print("Monthly returns for SPY:")
print(monthly_returns.tail(12))

Monthly returns for SPY:
Date
2023-12-31    0.045655
2024-01-31    0.015926
2024-02-29    0.052187
2024-03-31    0.032702
2024-04-30   -0.040320
2024-05-31    0.050580
2024-06-30    0.035280
2024-07-31    0.012109
2024-08-31    0.023366
2024-09-30    0.021005
2024-10-31   -0.008924
2024-11-30    0.059634
Freq: ME, Name: Adj Close, dtype: float64


**Summary of Aggregation Methods:**

| Data Type | Correct Method | Why |
|-----------|----------------|-----|
| Prices | `.last()` | You want the period-ending price |
| Volume | `.sum()` | Volume accumulates over time |
| Returns | `.prod()` on (1 + r) | Returns compound, not add |
| High Price | `.max()` | You want the highest price in the period |
| Low Price | `.min()` | You want the lowest price in the period |
| Open Price | `.first()` | You want the period-starting price |

You can also use `.agg()` to apply different aggregation methods to different columns in a single operation:

In [110]:
# Using .agg() to apply different methods to different columns
df_ohlcv = df_spy[['Open', 'High', 'Low', 'Adj Close', 'Volume']].copy()

# Resample to monthly with appropriate aggregations
df_monthly = df_ohlcv.resample('ME').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Adj Close': 'last',
    'Volume': 'sum'
})

print(df_monthly.tail(6))

Price             Open        High         Low   Adj Close      Volume
Date                                                                  
2024-06-30  529.020020  550.280029  522.599976  530.317566   888367600
2024-07-31  545.630005  565.159973  537.450012  536.739197  1038465500
2024-08-31  552.570007  564.200012  510.269989  549.280457  1244599000
2024-09-30  560.469971  574.710022  539.440002  560.817871  1044988300
2024-10-31  573.400024  586.119995  565.270020  555.813232   976068800
2024-11-30  571.320007  603.349976  567.890015  588.958496   901760600



**Section Summary**

You have now learned the essential time-series operations in Pandas:

- `.rolling()` for moving window calculations like moving averages and rolling volatility
- `.shift()` for accessing previous or future values, which is critical for avoiding lookahead bias
- `.cumsum()` and `.cumprod()` for cumulative operations, with `.cumprod()` being the standard for equity curves
- `.resample()` for changing data frequency, with the critical lesson that different data types require different aggregation methods

These methods are the building blocks of trading strategies. In the chapters ahead, you will use them constantly to build signals, calculate indicators, and evaluate performance.

---

### 1.15 Combining and Reshaping DataFrames

This section introduces essential methods for combining and reshaping DataFrames in Pandas. These are critical for organizing and analyzing financial datasets, especially when merging, aligning, or restructuring data from multiple sources.

We cannot stress enough how important this section is! While it might seem daunting at first, these are techniques which you must master!

We will explain them in very basic terms, but feel free to look at the pandas documentation or supplement explanations using a generative AI tool.

#### The Jagged Edge Problem

Before we dive into the specific methods, we need to understand the core problem they solve.

In the real world, financial data rarely aligns perfectly. Different assets trade on different exchanges with different holiday calendars. The US stock market is closed on July 4th, but Canadian markets are open. European markets close on different holidays than Asian markets. Even within the same country, some data sources might have missing days due to data errors.

This creates what we call the jagged edge problem. You have two time series that should align by date, but they have slightly different sets of dates.

Let's see this in action by downloading data for SPY (the S&P 500 ETF) and VIX (the volatility index). While both trade in the US, we can simulate the problem by examining real data and seeing how even small misalignments can cause issues.

In [111]:
# Download SPY and VIX data
# VIX is the CBOE Volatility Index - it measures market fear
import yfinance as yf

ticker_spy = 'SPY'
ticker_vix = '^VIX'
start_date = '2024-01-01'
end_date = '2024-06-30'

if USE_LIVE_DATA:
    df_spy_demo = yf.download(ticker_spy, start=start_date, end=end_date, auto_adjust=False)['Adj Close']
    df_vix_demo = yf.download(ticker_vix, start=start_date, end=end_date, auto_adjust=False)['Adj Close']
else:
    df_spy_demo = load_saved_data('ch02_nb01_spy_2024h1.parquet')['Adj Close']
    df_vix_demo = load_saved_data('ch02_nb01_vix_2024h1.parquet')['Adj Close']

# Let's check the shapes
print(f"\nSPY data points: {len(df_spy_demo)}")
print(f"VIX data points: {len(df_vix_demo)}")


SPY data points: 124
VIX data points: 124


In most cases, these two will have the same number of data points since they both trade on US exchanges. But let's verify that the dates actually match.

In [112]:
# Check if the indexes are identical
print(f"Are the indexes identical? {df_spy_demo.index.equals(df_vix_demo.index)}")

# Find any dates that are in one but not the other (Notice how we use .index!)
spy_dates = set(df_spy_demo.index)
vix_dates = set(df_vix_demo.index)

only_in_spy = spy_dates - vix_dates
only_in_vix = vix_dates - spy_dates

print(f"\nDates only in SPY: {only_in_spy}")
print(f"Dates only in VIX: {only_in_vix}")

Are the indexes identical? True

Dates only in SPY: set()
Dates only in VIX: set()


Depending on when you run this, you may or may not see differences. The point is that you cannot assume alignment.

To make sure we see the problem clearly, let's simulate a common real-world scenario: one dataset has a few dates that the other does not. This happens constantly when combining data from different sources.

In [113]:
# Create two DataFrames with slightly different dates
# Imagine df_canada is missing July 4th (US holiday when Canada trades)
# and df_us is missing July 1st (Canada Day when US trades)

dates_us = pd.to_datetime(['2024-06-28', '2024-07-01', '2024-07-02', '2024-07-03',
                            '2024-07-05', '2024-07-08', '2024-07-09'])
dates_canada = pd.to_datetime(['2024-06-28', '2024-07-02', '2024-07-03', '2024-07-04',
                                '2024-07-05', '2024-07-08', '2024-07-09'])

df_us = pd.DataFrame({'US_Price': [100, 101, 102, 103, 104, 105, 106]}, index=dates_us)
df_canada = pd.DataFrame({'Canada_Price': [50, 51, 52, 53, 54, 55, 56]}, index=dates_canada)

print("US Trading Dates:")
print(df_us)
print("\nCanada Trading Dates:")
print(df_canada)

US Trading Dates:
            US_Price
2024-06-28       100
2024-07-01       101
2024-07-02       102
2024-07-03       103
2024-07-05       104
2024-07-08       105
2024-07-09       106

Canada Trading Dates:
            Canada_Price
2024-06-28            50
2024-07-02            51
2024-07-03            52
2024-07-04            53
2024-07-05            54
2024-07-08            55
2024-07-09            56


Notice the differences:

- July 1st (Canada Day) is in the US data but not in the Canada data
- July 4th (US Independence Day) is in the Canada data but not in the US data

This is the jagged edge. Now let's see what happens when we try to combine these DataFrames using different methods.

---

#### 1.15.1 `pd.concat`

**What It Does**
- pd.concat combines multiple DataFrames along rows (axis=0) or columns (axis=1).

**When to Use It**
- Use when you want to stack DataFrames vertically (row-wise) or combine them horizontally (column-wise) without explicitly matching on an index or key.

**Parameters**
- objs: A list of DataFrames to concatenate.
- axis: Specifies the direction of concatenation:
 - 0 for rows (default).
 - 1 for columns.
- ignore_index: Resets the index in the concatenated DataFrame if True.
- keys: Adds hierarchical keys to identify the origin of each DataFrame.

pd.concat is the simplest way to combine different dataframes. But that simplicity comes at a cost. Pandas might not always know what you are trying to do. If you want to stack dataframes vertically, meaning add one dataframe to the bottom of the other, the column labels must be the exact same! Otherwise, you will see new columns added and there will be NaN.

If you want to add the second dataframe to the right of the first dataframe, the indexes must be exactly the same.

Let's start with a simple example and then we will demonstrate what happens when the labels aren't the same.

In [114]:
# Create sample DataFrames
df1 = pd.DataFrame({"Price": [100, 105], "Volume": [1000, 1200]}, index=["2023-01-01", "2023-01-02"])
df2 = pd.DataFrame({"Price": [110, 115], "Volume": [1500, 1600]}, index=["2023-01-03", "2023-01-04"])

# Concatenate along rows
df_combined = pd.concat([df1, df2])  # The order of the dataframes matter! in this case, you are adding df2 below df1

# Concatenate along columns
df_combined_columns = pd.concat([df1, df2], axis=1)  # The order of the dataframes matter! in this case, you are adding df2 to the right of df1

# Concatenate with hierarchical keys
df_with_keys = pd.concat([df1, df2], keys=["DF1", "DF2"])


In [115]:
# Let's examine the first one, stacked vertically
print(df_combined)
# You see that the rows of df2 are now below df1

            Price  Volume
2023-01-01    100    1000
2023-01-02    105    1200
2023-01-03    110    1500
2023-01-04    115    1600


In [116]:
# Let's examine the second one, stacked horizontally
print(df_combined_columns)
# You see that the columns of df2 are to the right of df1.
# also notice that we have columns with the same names!

            Price  Volume  Price  Volume
2023-01-01  100.0  1000.0    NaN     NaN
2023-01-02  105.0  1200.0    NaN     NaN
2023-01-03    NaN     NaN  110.0  1500.0
2023-01-04    NaN     NaN  115.0  1600.0


In [117]:
# We can assign a second level to the indexes with the keys parameter.
# But we will not do this very often, since it makes things complicated.
df_with_keys

Price  Volume
DF1 2023-01-01    100    1000
    2023-01-02    105    1200
DF2 2023-01-03    110    1500
    2023-01-04    115    1600

In [118]:
# Now let's address handling the case of axis=1 where the column names are the same

# Create sample DataFrames with duplicate column names
df1 = pd.DataFrame({"Price": [100, 105], "Volume": [1000, 1200]}, index=["2023-01-01", "2023-01-02"])
df2 = pd.DataFrame({"Price": [110, 115], "Volume": [1500, 1600]}, index=["2023-01-01", "2023-01-02"])

# Concatenate along columns with duplicate column names
df_combined = pd.concat([df1, df2], axis=1)
print(df_combined)


            Price  Volume  Price  Volume
2023-01-01    100    1000    110    1500
2023-01-02    105    1200    115    1600


Look very closely at the difference here to the first time we did this. Now the indexes are the same in both dfs, so we don't have any NaN. Which is great!

But we don't want duplicated names. So we can handle this in two ways.
1. Add a multiindex (add a level above price and volume that identifies which df it is).

In [119]:
df_combined = pd.concat([df1, df2], axis=1, keys=["DF1", "DF2"])
print(df_combined)


             DF1          DF2       
           Price Volume Price Volume
2023-01-01   100   1000   110   1500
2023-01-02   105   1200   115   1600


But this can complicate things later when we start doing `.loc` and `.iloc`

2. Rename the columns before or after concatenation

In [120]:
col_names_after = ['df1_price', 'df1_volume', 'df2_price', 'df2_volume']
df_combined.columns = col_names_after
df_combined

,df1_price,df1_volume,df2_price,df2_volume
2023-01-01,100,1000,110,1500
2023-01-02,105,1200,115,1600


Let's look at what can go wrong with pd.concat when you have mismatched indexes and labels.

First, think about the context. Imagine we have two pricing dataframes, and we want to create one dataframe that has just the Adj Close columns for each. This means that we want to stack them horizontally, so we can see the Adj Close of both tickers next to each other for the same dates.

But what if you have indexes (dates) that aren't the same? Imagine one is in Canada and the other is US and they have different trading holidays? Or if one index has dates as strings anf the other index is in datetime? These kinds of problems happen all the time.


In [121]:
# Create two DataFrames with mismatched indexes
df1 = pd.DataFrame({"PEP Price": [100, 105]}, index=["2023-01-01", "2023-01-02"])
df2 = pd.DataFrame({"WMT Price": [150, 160]}, index=["2023-01-03", "2023-01-04"])

# Concatenate along columns
df_combined = pd.concat([df1, df2], axis=1)
print(df_combined)


            PEP Price  WMT Price
2023-01-01      100.0        NaN
2023-01-02      105.0        NaN
2023-01-03        NaN      150.0
2023-01-04        NaN      160.0


You can see the NaN because those values don't exist. This produced a df with 4 rows when all we wanted was 2.

Now let's look at mismatched columns when the intention is to stack them vertically.

In [122]:
# Create two DataFrames with mismatched column names
df1 = pd.DataFrame({"Price": [100, 105]})
df2 = pd.DataFrame({"Cost": [1500, 1600]})

# Concatenate along rows
df_combined = pd.concat([df1, df2], axis=0)
print(df_combined)


   Price    Cost
0  100.0     NaN
1  105.0     NaN
0    NaN  1500.0
1    NaN  1600.0


You could see that because the columns have different names, our output has 2 columns when we only wanted one.

In [123]:
# This is what we wanted
df1 = pd.DataFrame({"Price": [100, 105]})
df2 = pd.DataFrame({"Price": [1500, 1600]})

# Concatenate along rows
df_combined = pd.concat([df1, df2], axis=0)
print(df_combined)


   Price
0    100
1    105
0   1500
1   1600


#### 1.15.2 The Alignment Failure Demo

Now let's see what happens when we try to combine our misaligned US and Canada DataFrames using `pd.concat` with `axis=1`. This is a mistake that many beginners make, and it can silently corrupt your analysis.

In [124]:
# Attempt to combine the two DataFrames horizontally using pd.concat
df_concat_fail = pd.concat([df_us, df_canada], axis=1)

print("Result of pd.concat with axis=1 on misaligned dates:")
print(df_concat_fail)

Result of pd.concat with axis=1 on misaligned dates:
            US_Price  Canada_Price
2024-06-28     100.0          50.0
2024-07-01     101.0           NaN
2024-07-02     102.0          51.0
2024-07-03     103.0          52.0
2024-07-04       NaN          53.0
2024-07-05     104.0          54.0
2024-07-08     105.0          55.0
2024-07-09     106.0          56.0


Look at the result carefully.

Pandas kept all the dates from both DataFrames, but now we have NaN values scattered throughout:

- July 1st has a US price but NaN for Canada (because Canada was closed)
- July 4th has a Canada price but NaN for US (because the US was closed)

This is a union of the two indexes. Pandas did not throw an error. It did not warn you. It just silently introduced missing data into your DataFrame.

Now imagine you calculate the correlation between US and Canada prices, or you try to compute a spread between them. Those NaN values will either cause errors or, worse, give you misleading results.

**This is why you cannot blindly use `pd.concat` for combining time-series data.**

In [125]:
# How many NaN values did we create?
print(f"Total NaN values: {df_concat_fail.isna().sum().sum()}")
print(f"\nNaN values per column:")
print(df_concat_fail.isna().sum())

Total NaN values: 2

NaN values per column:
US_Price        1
Canada_Price    1
dtype: int64


Two NaN values were introduced silently. In a small example like this, it is easy to spot. In a DataFrame with thousands of rows and dozens of columns, these problems can go unnoticed until they corrupt your backtest results.

The solution is to use `df.join()` with the appropriate `how` parameter, which gives you explicit control over how misaligned indexes are handled.




---

#### 1.15.3 `df.join`


The df.join() method in Pandas is used to combine two DataFrames based on their index values. This is particularly useful when you have related datasets (e.g., stock prices and trading volumes) that share the same index (such as dates) and you want to merge them into a single DataFrame.

**What Does df.join() Do?**
- Joins two or more DataFrames together by aligning their rows based on the index.
- Adds the columns of one DataFrame to another, creating a single DataFrame.
- Allows for flexibility in handling rows that do not have matching indices (via how).

**Parameters of df.join()**
1. other:

- The DataFrame to join with the current DataFrame.

2. how:

- Determines how rows are matched between the two DataFrames:
 - 'inner': Keeps only rows with indices that are common to both DataFrames.
 - 'outer': Keeps all rows from both DataFrames, filling missing values with NaN.
 - 'left': Keeps all rows from the calling DataFrame, filling with NaN for missing rows from the other DataFrame.
 - 'right': Keeps all rows from the other DataFrame, filling with NaN for missing rows from the calling DataFrame.

3. on (Optional):

- Specifies the column(s) to join on if your DataFrame does not have an index to align on. (Typically not used with beginners.)

4. lsuffix and rsuffix:

- Add suffixes to overlapping column names from the left (calling DataFrame) and right (other DataFrame) to avoid conflicts.

Let's start with some examples. We know this seems confusing, but practice and understanding when to use it will help.

Example 1: Basic join with matching indexes

**The Fix:** Solving the Jagged Edge with df.join()

Now let's return to our US and Canada example and see how `df.join()` handles the misalignment problem properly.

The key is the `how` parameter. It forces you to make an explicit decision about what to do with mismatched dates. No silent failures. No hidden NaN values appearing unexpectedly.

##### Inner Join

In [126]:
# Inner join: Keep only dates that exist in BOTH DataFrames
df_inner = df_us.join(df_canada, how='inner')

print("Inner Join Result (only matching dates):")
print(df_inner)
print(f"\nRows: {len(df_inner)}")
print(f"NaN values: {df_inner.isna().sum().sum()}")

Inner Join Result (only matching dates):
            US_Price  Canada_Price
2024-06-28       100            50
2024-07-02       102            51
2024-07-03       103            52
2024-07-05       104            54
2024-07-08       105            55
2024-07-09       106            56

Rows: 6
NaN values: 0


The inner join kept only the six dates that exist in both DataFrames. July 1st and July 4th were dropped because they did not have data for both countries.

This is the safest option when you need complete data for all columns. There are no NaN values. Every calculation you perform will have valid data on both sides.

**When to use inner join:** When you need both values to exist for your analysis. For example, calculating the correlation between two assets, or computing a spread that requires both prices.

##### Outer Join

An outer join keeps all rows from both DataFrames. If a date exists in one DataFrame but not the other, Pandas fills the missing values with NaN.

This is the most inclusive option. You lose nothing, but you must be prepared to handle the missing data afterward.

In [127]:
# Outer join: Keep ALL dates from BOTH DataFrames
df_outer = df_us.join(df_canada, how='outer')

print("Outer Join Result (all dates from both):")
print(df_outer)
print(f"\nRows: {len(df_outer)}")
print(f"NaN values: {df_outer.isna().sum().sum()}")

Outer Join Result (all dates from both):
            US_Price  Canada_Price
2024-06-28     100.0          50.0
2024-07-01     101.0           NaN
2024-07-02     102.0          51.0
2024-07-03     103.0          52.0
2024-07-04       NaN          53.0
2024-07-05     104.0          54.0
2024-07-08     105.0          55.0
2024-07-09     106.0          56.0

Rows: 8
NaN values: 2


The outer join kept all eight dates from both DataFrames. Notice the two NaN values:

- July 1st has a US price but NaN for Canada (Canada Day holiday)
- July 4th has a Canada price but NaN for US (US Independence Day holiday)

This is the same result we saw with `pd.concat(axis=1)`. The difference is intent. With `df.join(how='outer')`, you are explicitly choosing to keep everything and you know you will need to handle the NaN values later, perhaps with `.fillna()` or `.ffill()`.

**When to use outer join:** When you want to preserve all history from both DataFrames and will handle missing values explicitly. For example, when building a master calendar of all trading days across multiple markets.

##### Left Join

A left join keeps all rows from the left DataFrame (the one calling `.join()`) and matches rows from the right DataFrame where possible. If a date exists in the left DataFrame but not the right, Pandas fills the missing value with NaN.

This is the default behavior of `df.join()` and is the most common choice in practice. You pick one DataFrame as your master calendar and align everything else to it.

In [128]:
# Left join: Keep all dates from the LEFT (calling) DataFrame
df_left = df_us.join(df_canada, how='left')

print("Left Join Result (all US dates, match Canada where possible):")
print(df_left)
print(f"\nRows: {len(df_left)}")
print(f"NaN values: {df_left.isna().sum().sum()}")

Left Join Result (all US dates, match Canada where possible):
            US_Price  Canada_Price
2024-06-28       100          50.0
2024-07-01       101           NaN
2024-07-02       102          51.0
2024-07-03       103          52.0
2024-07-05       104          54.0
2024-07-08       105          55.0
2024-07-09       106          56.0

Rows: 7
NaN values: 1


The left join kept all seven dates from the US DataFrame and matched Canada data where available.

- July 1st shows NaN for Canada because Canada was closed that day
- July 4th was dropped entirely because it is not in the US calendar

This is the most common choice in practice. You designate one DataFrame as your primary source and align everything else to it.

**When to use left join:** When you have a primary dataset (like your main trading calendar or strategy signal dates) and you want to add supplementary data without changing your date index.

##### Right Join

A right join is the opposite of a left join. It keeps all rows from the right DataFrame (the one passed into `.join()`) and matches rows from the left DataFrame where possible.

In practice, you will rarely use this. If you want to align to the right DataFrame, it is clearer to simply reverse the order and use a left join. But it is important to know it exists.

In [129]:
# Right join: Keep all dates from the RIGHT DataFrame
df_right = df_us.join(df_canada, how='right')

print("Right Join Result (all Canada dates, match US where possible):")
print(df_right)
print(f"\nRows: {len(df_right)}")
print(f"NaN values: {df_right.isna().sum().sum()}")

Right Join Result (all Canada dates, match US where possible):
            US_Price  Canada_Price
2024-06-28     100.0            50
2024-07-02     102.0            51
2024-07-03     103.0            52
2024-07-04       NaN            53
2024-07-05     104.0            54
2024-07-08     105.0            55
2024-07-09     106.0            56

Rows: 7
NaN values: 1


The right join kept all seven dates from the Canada DataFrame and matched US data where available.

- July 4th shows NaN for US because the US was closed that day
- July 1st was dropped entirely because it is not in the Canada calendar

Notice this is the mirror image of the left join. In practice, if you wanted this result, you would more likely write:

`df_canada.join(df_us, how='left')`

This achieves the same result but is easier to read because the primary DataFrame comes first.

#### 1.15.4 pd.merge()

The `pd.merge()` function joins DataFrames based on column values rather than the index. If you have experience with SQL, this will feel familiar. It is the equivalent of a SQL JOIN.

**What It Does:**

- Combines two DataFrames by matching values in one or more columns
- Does not rely on the index for alignment
- Provides full control over join type (inner, outer, left, right)

**When to Use It:**

- When your join key is a column, not the index
- When matching transactional data to reference tables (like matching trade records to a table of ticker symbols)
- When working with relational data structures

Let's create a reference table with country metadata and merge it with our price data.

In [130]:
# First, let's reshape our US/Canada data into a long format for this demo
# We will create a simple transactions table

transactions = pd.DataFrame({
    'Date': ['2024-07-02', '2024-07-03', '2024-07-05', '2024-07-02', '2024-07-03', '2024-07-05'],
    'Country': ['US', 'US', 'US', 'Canada', 'Canada', 'Canada'],
    'Price': [102, 103, 104, 51, 52, 54]
})

# Create a reference table with country metadata
country_info = pd.DataFrame({
    'Country': ['US', 'Canada', 'UK'],
    'Currency': ['USD', 'CAD', 'GBP'],
    'Exchange': ['NYSE', 'TSX', 'LSE']
})

print("Transactions Table:")
print(transactions)
print("\nCountry Reference Table:")
print(country_info)

Transactions Table:
         Date Country  Price
0  2024-07-02      US    102
1  2024-07-03      US    103
2  2024-07-05      US    104
3  2024-07-02  Canada     51
4  2024-07-03  Canada     52
5  2024-07-05  Canada     54

Country Reference Table:
  Country Currency Exchange
0      US      USD     NYSE
1  Canada      CAD      TSX
2      UK      GBP      LSE


In [131]:
# Merge the transactions with country info based on the Country column
df_merged = pd.merge(transactions, country_info, on='Country', how='left')

print("Merged Result:")
print(df_merged)

Merged Result:
         Date Country  Price Currency Exchange
0  2024-07-02      US    102      USD     NYSE
1  2024-07-03      US    103      USD     NYSE
2  2024-07-05      US    104      USD     NYSE
3  2024-07-02  Canada     51      CAD      TSX
4  2024-07-03  Canada     52      CAD      TSX
5  2024-07-05  Canada     54      CAD      TSX


The merge matched each row in the transactions table to the corresponding row in the country reference table based on the Country column. Each transaction now has the Currency and Exchange information attached.

Notice a few things:

1. We used `on='Country'` to specify the column to match on. Both DataFrames must have a column with this name.

2. The UK row in the reference table was not included because there were no UK transactions. This is because we used `how='left'`, which keeps all rows from the left DataFrame (transactions).

3. Unlike `df.join()`, the merge did not use the index at all. It matched purely on column values.

**Key Parameters:**

- `on`: The column name to join on (must exist in both DataFrames)
- `left_on` and `right_on`: Use these when the column names differ between DataFrames
- `how`: Same options as `df.join()` (inner, outer, left, right)

#### 1.15.5 Decision Framework: concat vs join vs merge

You now have three tools for combining DataFrames. Here is when to use each one.

**Use `pd.concat()` when:**

- Stacking DataFrames vertically (adding rows with `axis=0`)
- Combining DataFrames that you know have identical indexes or columns
- Building a list of DataFrames in a loop and combining them at the end
- Speed is critical and you have already verified alignment

**Use `df.join()` when:**

- Combining time-series data that is indexed by date
- You need explicit control over how misaligned dates are handled
- Adding columns from one DataFrame to another based on index alignment
- Working with financial data where different assets may have different trading calendars

**Use `pd.merge()` when:**

- Joining on column values rather than the index (like SQL joins)
- Combining DataFrames where the join key is not the index
- Working with relational data (like matching trades to a reference table of symbols)

For most financial time-series work, `df.join()` with an explicit `how` parameter is the safest and most appropriate choice.



---


#### 1.15.6 df.pivot()

The `df.pivot()` method reshapes your DataFrame from long format to wide format. It takes values from rows and spreads them across columns.

**What It Does:**

- Converts unique values from one column into new column headers
- Reorganizes the data structure without aggregation
- Creates a wider DataFrame with fewer rows

**When to Use It:**

- When you have data in long format and need it in wide format for analysis
- When you want each unique category to become its own column
- When preparing data for correlation analysis or side-by-side comparison

Let's take our transactions table and pivot it so that US and Canada prices become separate columns.

In [132]:
# Pivot the transactions table from long to wide format
df_pivoted = transactions.pivot(index='Date', columns='Country', values='Price')

print("Original long form:")
print(transactions)

print("\nPivoted Result (long to wide):")
print(df_pivoted)

Original long form:
         Date Country  Price
0  2024-07-02      US    102
1  2024-07-03      US    103
2  2024-07-05      US    104
3  2024-07-02  Canada     51
4  2024-07-03  Canada     52
5  2024-07-05  Canada     54

Pivoted Result (long to wide):
Country     Canada   US
Date                   
2024-07-02      51  102
2024-07-03      52  103
2024-07-05      54  104


The pivot transformed our long table into a wide table:

- The Date column became the index
- The unique values in Country (Canada, US) became column headers
- The Price values filled in the cells

This is now in the same format as when we joined df_us and df_canada earlier. We have gone from a table with 6 rows and 3 columns to a table with 3 rows and 2 columns.

**Key Parameters:**

- `index`: The column to use as the new row index
- `columns`: The column whose unique values become new column headers
- `values`: The column containing the values to fill the cells

**Important:** The `df.pivot()` method requires that each combination of index and columns be unique. If you have duplicate combinations, you will get an error. In that case, use `.pivot_table()` which can aggregate duplicates.

#### 1.15.7 pd.melt()

The `pd.melt()` function is the reverse of pivot. It takes a wide DataFrame and reshapes it into long format by unpivoting columns into rows.

**What It Does:**

- Converts column headers into row values
- Creates a longer DataFrame with fewer columns
- Gathers multiple columns into a single column

**When to Use It:**

- When you have data in wide format and need it in long format for analysis
- When preparing data for visualization libraries that prefer long format
- When you need to stack multiple columns into a single column for grouping or aggregation

Let's take our pivoted DataFrame and melt it back to the original long format.

In [133]:
# First, reset the index so Date becomes a column again
df_to_melt = df_pivoted.reset_index()

print("Before melt:")
print(df_to_melt)

# Melt the DataFrame back to long format
df_melted = pd.melt(df_to_melt, id_vars='Date', var_name='Country', value_name='Price')

print("\nAfter melt (wide to long):")
print(df_melted)

Before melt:
Country        Date  Canada   US
0        2024-07-02      51  102
1        2024-07-03      52  103
2        2024-07-05      54  104

After melt (wide to long):
         Date Country  Price
0  2024-07-02  Canada     51
1  2024-07-03  Canada     52
2  2024-07-05  Canada     54
3  2024-07-02      US    102
4  2024-07-03      US    103
5  2024-07-05      US    104


The melt transformed our wide table back into long format:

- The Date column stayed in place (because we specified it as `id_vars`)
- The column headers (Canada, US) became values in a new Country column
- The cell values became a new Price column

We have now completed the round-trip: long → wide → long.

**Key Parameters:**

- `id_vars`: Columns to keep as identifiers (they stay in place)
- `var_name`: Name for the new column that holds the former column headers
- `value_name`: Name for the new column that holds the values

This is a powerful tool for data transformation. Many visualization libraries like Seaborn and Plotly prefer long format data, so you will often need to melt wide DataFrames before plotting.



---
#### 1.15.8 .pivot_table()

The `.pivot_table()` method is like `df.pivot()` but with aggregation built in. When you have duplicate combinations of index and columns, `.pivot_table()` can handle them by applying an aggregation function.

**What It Does:**

- Reshapes data from long to wide format (like pivot)
- Aggregates values when there are duplicates
- Provides summary statistics across groups

**When to Use It:**

- When your data has duplicate index-column combinations
- When you need to aggregate while reshaping (sum, mean, count, etc.)
- When creating summary tables for analysis or reporting

Let's create a larger transactions dataset with multiple trades per day and use pivot_table to summarize it.


In [134]:
# Create a dataset with multiple transactions per day
transactions_multi = pd.DataFrame({
    'Date': ['2024-07-02', '2024-07-02', '2024-07-02', '2024-07-03', '2024-07-03', '2024-07-03',
             '2024-07-02', '2024-07-02', '2024-07-03', '2024-07-03'],
    'Country': ['US', 'US', 'US', 'US', 'US', 'US',
                'Canada', 'Canada', 'Canada', 'Canada'],
    'Volume': [1000, 1500, 800, 2000, 1200, 900,
               500, 700, 600, 400]
})

print("Transactions with multiple entries per day:")
print(transactions_multi)

Transactions with multiple entries per day:
         Date Country  Volume
0  2024-07-02      US    1000
1  2024-07-02      US    1500
2  2024-07-02      US     800
3  2024-07-03      US    2000
4  2024-07-03      US    1200
5  2024-07-03      US     900
6  2024-07-02  Canada     500
7  2024-07-02  Canada     700
8  2024-07-03  Canada     600
9  2024-07-03  Canada     400


In [135]:
# Pivot table with aggregation: sum the volume by date and country
df_pivot_table = transactions_multi.pivot_table(
    index='Date',
    columns='Country',
    values='Volume',
    aggfunc='sum'
)

print("Pivot Table Result (total volume by date and country):")
print(df_pivot_table)

Pivot Table Result (total volume by date and country):
Country     Canada    US
Date                    
2024-07-02    1200  3300
2024-07-03    1000  4100


The pivot table aggregated the multiple transactions into a single summary value for each date and country combination.

- July 2nd had three US transactions (1000, 1500, 800) which summed to 3300
- July 2nd had two Canada transactions (500, 700) which summed to 1200

If we had tried to use `df.pivot()` on this data, we would have received an error because of the duplicate date-country combinations.

**Key Parameters:**

- `index`: The column to use as row labels
- `columns`: The column whose unique values become column headers
- `values`: The column to aggregate
- `aggfunc`: The aggregation function (sum, mean, count, min, max, etc.)

You can also pass multiple aggregation functions:

In [136]:
# Pivot table with multiple aggregation functions
df_pivot_multi = transactions_multi.pivot_table(
    index='Date',
    columns='Country',
    values='Volume',
    aggfunc=['sum', 'mean', 'count']
)

print("Pivot Table with multiple aggregations:")
print(df_pivot_multi)

Pivot Table with multiple aggregations:
              sum         mean               count   
Country    Canada    US Canada           US Canada US
Date                                                 
2024-07-02   1200  3300  600.0  1100.000000      2  3
2024-07-03   1000  4100  500.0  1366.666667      2  3


This gives you a comprehensive summary in one operation. Each aggregation function creates its own set of columns.

The `.pivot_table()` method is extremely useful for exploratory data analysis and creating summary reports. It combines reshaping and aggregation into a single, powerful tool.

---

In [137]:
# Download prices for SPY
if USE_LIVE_DATA:
    df_spy = yf.download('SPY', start_date, end_date, auto_adjust=False)
else:
    df_spy = load_saved_data('ch02_nb01_spy_2024h1.parquet')

# Remove the multi index
df_spy.columns = df_spy.columns.get_level_values(0)

df_spy.head()


Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2024-01-02,457.672760,472.649994,473.670013,470.489990,472.160004,123623700
2024-01-03,453.935059,468.790009,471.190002,468.170013,470.429993,103585900
2024-01-04,452.472839,467.279999,470.959991,467.049988,468.299988,84232200
2024-01-05,453.092621,467.920013,470.440002,466.429993,467.489990,86118900
2024-01-08,459.560944,474.600006,474.750000,468.299988,468.429993,74879100


In [138]:
# Export the dataframe to a CSV
# Normally, you are fine with just:
df_spy.to_csv('spy_prices.csv')  # just make sure you don't forger the .csv!

# If you look at the file explorer to our left in Colab, you will now see the file.

Now let's import it back in. Take a look at the key parameters that should be specified. Also note the versatility of this and how much of the heavy lifting Pandas can do for you.

```
df_spy_imported = pd.read_csv(
    "spy_data.csv",   # File path or name
    index_col="Date", # Specify which column to use as the index
    header=0,         # Specify the row number of the column headers (default is 0)
    parse_dates=True, # Convert date strings into datetime objects
    na_values=["NA", "NaN"], # Define which values should be interpreted as NaN
    keep_default_na=True # Include Pandas' default NaN identifiers
)
```
What is really cool, is that you can tell Python how NaN are represented in your spreadsheet! It can be #N/A (the default Excel NaN) or even blanks " ".

In [139]:
# Note that any files you import into Colab must be found in the file explorer to your left!
# You can drag and drop any files from your local into Colab.
# Just note that those files are only here as long as Colab is running. Once your session ends
# the file system is gone. Make sure to download any files you exported!

df_spy_imported = pd.read_csv(
    "spy_prices.csv",   # File path or name
    index_col="Date", # Specify which column to use as the index
    header=0,         # Specify the row number of the column headers (default is 0)
    parse_dates=True, # Convert date strings into datetime objects
    na_values=["NA", "NaN"], # Define which values should be interpreted as NaN
    keep_default_na=True # Include Pandas' default NaN identifiers
)

df_spy_imported.head()

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2024-01-02,457.672760,472.649994,473.670013,470.489990,472.160004,123623700
2024-01-03,453.935059,468.790009,471.190002,468.170013,470.429993,103585900
2024-01-04,452.472839,467.279999,470.959991,467.049988,468.299988,84232200
2024-01-05,453.092621,467.920013,470.440002,466.429993,467.489990,86118900
2024-01-08,459.560944,474.600006,474.750000,468.299988,468.429993,74879100


And we have our data imported! Before we move on, we should check to see what format our index is in. We want it in datetime.

In [140]:
df_spy_imported.index

DatetimeIndex(['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05',
               '2024-01-08', '2024-01-09', '2024-01-10', '2024-01-11',
               '2024-01-12', '2024-01-16',
               ...
               '2024-06-14', '2024-06-17', '2024-06-18', '2024-06-20',
               '2024-06-21', '2024-06-24', '2024-06-25', '2024-06-26',
               '2024-06-27', '2024-06-28'],
              dtype='datetime64[ns]', name='Date', length=124, freq=None)

And it is! This is because we set `parse_dates` to True.

In [141]:
# Save our data as a pkl file
df_spy.to_pickle('spy_prices.pkl')

# Notice that we don't to specify any of parameters about index and header!
# It's all saved in the file format.

In [142]:
df_spy_imported = pd.read_pickle("spy_prices.pkl")
df_spy_imported.head()

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2024-01-02,457.672760,472.649994,473.670013,470.489990,472.160004,123623700
2024-01-03,453.935059,468.790009,471.190002,468.170013,470.429993,103585900
2024-01-04,452.472839,467.279999,470.959991,467.049988,468.299988,84232200
2024-01-05,453.092621,467.920013,470.440002,466.429993,467.489990,86118900
2024-01-08,459.560944,474.600006,474.750000,468.299988,468.429993,74879100


And that's it! You now know how to save and import data, which is really important when dealing with large datasets.

## 2.0 DataFrame Math

Now that you can create, manipulate, and combine DataFrames, you need to perform calculations on them. Pandas provides arithmetic operations that automatically align data by index and broadcast across different shapes.

This section covers the essential math operations you will use constantly: calculating returns, subtracting benchmarks, applying weights, and normalizing data.

### 2.1 Arithmetic Operations


#### 2.1.1 Basic Operations: +, -, *, /

Pandas supports standard arithmetic operators between DataFrames, Series, and scalar values. These operations work element-wise, meaning each value is paired with its corresponding value based on index alignment.

In [143]:
# Create sample price data for two stocks
dates = pd.date_range('2024-01-01', periods=5, freq='D')

prices = pd.DataFrame({
    'AAPL': [185.50, 186.25, 184.75, 187.00, 188.50],
    'MSFT': [375.00, 377.50, 374.25, 378.00, 380.25]
}, index=dates)

print("Stock Prices:")
print(prices)

Stock Prices:
              AAPL    MSFT
2024-01-01  185.50  375.00
2024-01-02  186.25  377.50
2024-01-03  184.75  374.25
2024-01-04  187.00  378.00
2024-01-05  188.50  380.25


In [144]:
# Scalar operations - apply to every element
prices_with_fee = prices + 0.01  # Add transaction cost
prices_doubled = prices * 2      # Double all values

print("Prices plus $0.01 fee:")
print(prices_with_fee)
print("\nPrices doubled:")
print(prices_doubled)

Prices plus $0.01 fee:
              AAPL    MSFT
2024-01-01  185.51  375.01
2024-01-02  186.26  377.51
2024-01-03  184.76  374.26
2024-01-04  187.01  378.01
2024-01-05  188.51  380.26

Prices doubled:
             AAPL   MSFT
2024-01-01  371.0  750.0
2024-01-02  372.5  755.0
2024-01-03  369.5  748.5
2024-01-04  374.0  756.0
2024-01-05  377.0  760.5


When you perform arithmetic with a scalar value, Pandas applies the operation to every element in the DataFrame. This is broadcasting at work. The single value is stretched to match the shape of the DataFrame.

This pattern is useful for adding transaction costs, converting units, or applying a flat adjustment across all values.

#### 2.1.2 Method Equivalents: .add(), .sub(), .mul(), .div()

Pandas provides method versions of each arithmetic operator. These methods offer more control, particularly when working with DataFrames and Series of different shapes.

In [145]:
# These pairs are equivalent
result1 = prices + 10
result2 = prices.add(10)

result3 = prices - 5
result4 = prices.sub(5)

result5 = prices * 2
result6 = prices.mul(2)

result7 = prices / 100
result8 = prices.div(100)

print("prices + 10 equals prices.add(10):", result1.equals(result2))
print("prices - 5 equals prices.sub(5):", result3.equals(result4))
print("prices * 2 equals prices.mul(2):", result5.equals(result6))
print("prices / 100 equals prices.div(100):", result7.equals(result8))

prices + 10 equals prices.add(10): True
prices - 5 equals prices.sub(5): True
prices * 2 equals prices.mul(2): True
prices / 100 equals prices.div(100): True


For scalar operations, the operators and methods are interchangeable. Use whichever you find more readable.

The methods become essential when you need to specify an axis for alignment, which we will see in the next section.

#### 2.1.3 The axis Parameter: Aligning Rows vs Columns

When performing arithmetic between a DataFrame and a Series, Pandas needs to know how to align them. The `axis` parameter controls this.

In [146]:
# Create a DataFrame of daily returns for 3 stocks
returns = pd.DataFrame({
    'AAPL': [0.02, -0.01, 0.03, -0.02, 0.01],
    'MSFT': [0.01, 0.02, -0.01, 0.03, 0.02],
    'GOOGL': [0.03, -0.02, 0.02, 0.01, -0.01]
}, index=pd.date_range('2024-01-01', periods=5, freq='D'))

# Daily risk-free rate (same for all stocks on each day)
risk_free = pd.Series([0.0001, 0.0001, 0.0002, 0.0001, 0.0002],
                       index=returns.index)

print("Daily Returns:")
print(returns)
print("\nDaily Risk-Free Rate:")
print(risk_free)

Daily Returns:
            AAPL  MSFT  GOOGL
2024-01-01  0.02  0.01   0.03
2024-01-02 -0.01  0.02  -0.02
2024-01-03  0.03 -0.01   0.02
2024-01-04 -0.02  0.03   0.01
2024-01-05  0.01  0.02  -0.01

Daily Risk-Free Rate:
2024-01-01    0.0001
2024-01-02    0.0001
2024-01-03    0.0002
2024-01-04    0.0001
2024-01-05    0.0002
Freq: D, dtype: float64


In [147]:
# Subtract risk-free rate from each stock's return
# axis=0 means align along the rows (index); think of it as by months.
excess_returns = returns.sub(risk_free, axis=0)

print("Excess Returns (returns minus risk-free rate):")
print(excess_returns)

Excess Returns (returns minus risk-free rate):
              AAPL    MSFT   GOOGL
2024-01-01  0.0199  0.0099  0.0299
2024-01-02 -0.0101  0.0199 -0.0201
2024-01-03  0.0298 -0.0102  0.0198
2024-01-04 -0.0201  0.0299  0.0099
2024-01-05  0.0098  0.0198 -0.0102


The `axis=0` parameter tells Pandas to align the Series with the DataFrame's index (the dates). On each date, the risk-free rate is subtracted from every stock's return.

Without specifying `axis=0`, Pandas would try to align by columns and fail because the Series index (dates) does not match the DataFrame columns (stock tickers).

This is the Pandas equivalent of NumPy broadcasting. Pandas handles the shape matching automatically based on index alignment.

### 2.2 Alignment and Broadcasting

One of Pandas' most powerful features is automatic index alignment. When you perform arithmetic between two DataFrames or Series, Pandas matches values by their index labels, not by position.

This behavior is different from NumPy, which aligns strictly by position. Understanding how Pandas alignment works will save you from subtle bugs.

#### 2.2.1 How Pandas Aligns by Index

When two DataFrames share the same index, arithmetic works exactly as you would expect. Values are matched by their labels.

In [148]:
# Two DataFrames with identical indices
dates = pd.date_range('2024-01-01', periods=4, freq='D')

portfolio_a = pd.DataFrame({
    'value': [100000, 101500, 100800, 102300]
}, index=dates)

portfolio_b = pd.DataFrame({
    'value': [50000, 50500, 50250, 51000]
}, index=dates)

print("Portfolio A:")
print(portfolio_a)
print("\nPortfolio B:")
print(portfolio_b)

Portfolio A:
             value
2024-01-01  100000
2024-01-02  101500
2024-01-03  100800
2024-01-04  102300

Portfolio B:
            value
2024-01-01  50000
2024-01-02  50500
2024-01-03  50250
2024-01-04  51000


In [149]:
# Combined portfolio value
total_value = portfolio_a + portfolio_b

print("Combined Portfolio Value:")
print(total_value)

Combined Portfolio Value:
             value
2024-01-01  150000
2024-01-02  152000
2024-01-03  151050
2024-01-04  153300


Pandas matched each date in Portfolio A with the same date in Portfolio B and added the values. This is intuitive when indices match perfectly.

The real question is what happens when they do not match.

#### 2.2.2 What Happens with Mismatched Indices

Financial data often has mismatched indices. Different exchanges have different trading calendars. Some assets do not trade on certain days. When you perform arithmetic on DataFrames with mismatched indices, Pandas fills unmatched positions with NaN.

In [150]:
# US stock trades Monday through Friday
us_dates = pd.to_datetime(['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05'])
us_stock = pd.DataFrame({
    'return': [0.01, -0.02, 0.015, 0.008]
}, index=us_dates)

# Canadian stock has a holiday on Jan 3
ca_dates = pd.to_datetime(['2024-01-02', '2024-01-04', '2024-01-05'])
ca_stock = pd.DataFrame({
    'return': [0.012, 0.018, -0.005]
}, index=ca_dates)

print("US Stock Returns:")
print(us_stock)
print("\nCanadian Stock Returns:")
print(ca_stock)

US Stock Returns:
            return
2024-01-02   0.010
2024-01-03  -0.020
2024-01-04   0.015
2024-01-05   0.008

Canadian Stock Returns:
            return
2024-01-02   0.012
2024-01-04   0.018
2024-01-05  -0.005


In [151]:
# Calculate spread between US and Canadian returns
spread = us_stock - ca_stock

print("Return Spread (US minus Canada):")
print(spread)

Return Spread (US minus Canada):
            return
2024-01-02  -0.002
2024-01-03     NaN
2024-01-04  -0.003
2024-01-05   0.013


#### 2.2.3 Practical Example: Subtracting the Risk-Free Rate

Let us put alignment and broadcasting together in a common quant workflow: calculating excess returns for a portfolio of stocks.

In [152]:
# Monthly returns for 4 stocks over 6 months
dates = pd.date_range('2024-01-31', periods=6, freq='ME')

stock_returns = pd.DataFrame({
    'AAPL': [0.032, -0.015, 0.028, 0.041, -0.008, 0.022],
    'MSFT': [0.025, 0.018, -0.012, 0.035, 0.015, 0.019],
    'GOOGL': [0.041, -0.022, 0.033, 0.028, -0.015, 0.031],
    'AMZN': [0.018, 0.025, 0.015, 0.052, 0.008, -0.012]
}, index=dates)

# Monthly risk-free rate (1-month T-bill)
risk_free = pd.Series(
    [0.004, 0.004, 0.0042, 0.0042, 0.0043, 0.0043],
    index=dates
)

print("Monthly Stock Returns:")
print(stock_returns)
print("\nMonthly Risk-Free Rate:")
print(risk_free)

Monthly Stock Returns:
             AAPL   MSFT  GOOGL   AMZN
2024-01-31  0.032  0.025  0.041  0.018
2024-02-29 -0.015  0.018 -0.022  0.025
2024-03-31  0.028 -0.012  0.033  0.015
2024-04-30  0.041  0.035  0.028  0.052
2024-05-31 -0.008  0.015 -0.015  0.008
2024-06-30  0.022  0.019  0.031 -0.012

Monthly Risk-Free Rate:
2024-01-31    0.0040
2024-02-29    0.0040
2024-03-31    0.0042
2024-04-30    0.0042
2024-05-31    0.0043
2024-06-30    0.0043
Freq: ME, dtype: float64


In [153]:
# Calculate excess returns: stock return minus risk-free rate
excess_returns = stock_returns.sub(risk_free, axis=0)

print("Excess Returns:")
print(excess_returns)

Excess Returns:
              AAPL    MSFT   GOOGL    AMZN
2024-01-31  0.0280  0.0210  0.0370  0.0140
2024-02-29 -0.0190  0.0140 -0.0260  0.0210
2024-03-31  0.0238 -0.0162  0.0288  0.0108
2024-04-30  0.0368  0.0308  0.0238  0.0478
2024-05-31 -0.0123  0.0107 -0.0193  0.0037
2024-06-30  0.0177  0.0147  0.0267 -0.0163


Each stock's return is reduced by that month's risk-free rate. Pandas aligned the risk-free Series with the DataFrame's index and broadcast the subtraction across all four stock columns.

This is a one-line operation that would require a loop in many other languages. The combination of index alignment and broadcasting makes Pandas exceptionally powerful for financial calculations.

### 2.3 Common Patterns

Now that you understand how Pandas handles arithmetic, alignment, and broadcasting, let us look at patterns you will use repeatedly in quantitative finance.

#### 2.3.1 Normalizing Across Columns

Many machine learning models perform better when features are on the same scale. Normalization transforms your data so that different indicators can be compared directly.

The most common approach is z-score normalization: subtract the mean and divide by the standard deviation.

In [154]:
# Raw indicator values with very different scales
indicators = pd.DataFrame({
    'momentum': [0.05, 0.12, -0.03, 0.08, 0.15],      # Small decimals
    'volume_ratio': [1.2, 2.5, 0.8, 1.9, 3.1],        # Single digits
    'volatility': [15.5, 18.2, 12.8, 20.1, 22.5]      # Double digits
}, index=pd.date_range('2024-01-01', periods=5, freq='D'))

print("Raw Indicators (different scales):")
print(indicators)
print("\nMeans:")
print(indicators.mean())
print("\nStandard Deviations:")
print(indicators.std())

Raw Indicators (different scales):
            momentum  volume_ratio  volatility
2024-01-01      0.05           1.2        15.5
2024-01-02      0.12           2.5        18.2
2024-01-03     -0.03           0.8        12.8
2024-01-04      0.08           1.9        20.1
2024-01-05      0.15           3.1        22.5

Means:
momentum         0.074
volume_ratio     1.900
volatility      17.820
dtype: float64

Standard Deviations:
momentum        0.069498
volume_ratio    0.935414
volatility      3.802236
dtype: float64


In [155]:
# Z-score normalization: (value - mean) / std
normalized = (indicators - indicators.mean()) / indicators.std()

print("Normalized Indicators (z-scores):")
print(normalized)
print("\nVerify: means are now ~0:")
print(normalized.mean().round(10))
print("\nVerify: standard deviations are now 1:")
print(normalized.std())

Normalized Indicators (z-scores):
            momentum  volume_ratio  volatility
2024-01-01 -0.345333     -0.748331   -0.610167
2024-01-02  0.661888      0.641427    0.099941
2024-01-03 -1.496442     -1.175949   -1.320276
2024-01-04  0.086333      0.000000    0.599647
2024-01-05  1.093553      1.282854    1.230855

Verify: means are now ~0:
momentum        0.0
volume_ratio    0.0
volatility      0.0
dtype: float64

Verify: standard deviations are now 1:
momentum        1.0
volume_ratio    1.0
volatility      1.0
dtype: float64


Each column is now centered around zero with a standard deviation of one. A value of 1.5 in any column means that observation is 1.5 standard deviations above the mean for that indicator.

Notice what happened here: `indicators.mean()` returns a Series with three values (one per column). When you subtract this Series from the DataFrame, Pandas broadcasts the subtraction across all rows. The same happens with division by `indicators.std()`.

This is broadcasting working in the column direction by default. No loops, no manual alignment.

#### 2.3.2 Applying Weights to a Returns Matrix

Portfolio return is the weighted sum of individual asset returns. This calculation combines element-wise multiplication and aggregation.

In [156]:
# Monthly returns for a 4-stock portfolio
returns = pd.DataFrame({
    'AAPL': [0.032, -0.015, 0.028],
    'MSFT': [0.025, 0.018, -0.012],
    'GOOGL': [0.041, -0.022, 0.033],
    'AMZN': [0.018, 0.025, 0.015]
}, index=pd.date_range('2024-01-31', periods=3, freq='ME'))

# Portfolio weights (must sum to 1)
weights = pd.Series({
    'AAPL': 0.30,
    'MSFT': 0.25,
    'GOOGL': 0.25,
    'AMZN': 0.20
})

print("Monthly Returns:")
print(returns)
print("\nPortfolio Weights:")
print(weights)
print(f"\nWeights sum to: {weights.sum()}")

Monthly Returns:
             AAPL   MSFT  GOOGL   AMZN
2024-01-31  0.032  0.025  0.041  0.018
2024-02-29 -0.015  0.018 -0.022  0.025
2024-03-31  0.028 -0.012  0.033  0.015

Portfolio Weights:
AAPL     0.30
MSFT     0.25
GOOGL    0.25
AMZN     0.20
dtype: float64

Weights sum to: 1.0


In [157]:
# Step 1: Multiply each return by its weight
weighted_returns = returns.mul(weights, axis=1)

print("Weighted Returns:")
print(weighted_returns)

# Step 2: Sum across columns to get portfolio return
portfolio_returns = weighted_returns.sum(axis=1)

print("\nPortfolio Returns:")
print(portfolio_returns)

Weighted Returns:
              AAPL     MSFT    GOOGL    AMZN
2024-01-31  0.0096  0.00625  0.01025  0.0036
2024-02-29 -0.0045  0.00450 -0.00550  0.0050
2024-03-31  0.0084 -0.00300  0.00825  0.0030

Portfolio Returns:
2024-01-31    0.02970
2024-02-29   -0.00050
2024-03-31    0.01665
Freq: ME, dtype: float64


The `axis=1` parameter in `.mul()` tells Pandas to align the weights Series with the DataFrame's columns (the stock tickers). Each stock's return is multiplied by its corresponding weight.

The `.sum(axis=1)` then adds across the columns for each row, giving you the portfolio return for each month.

This two-step pattern (weight then sum) is fundamental to portfolio analysis. You will use it for calculating portfolio returns, risk contributions, and factor exposures.

#### 2.3.3 Handling Missing Data in Arithmetic

When indices do not align perfectly, Pandas fills missing values with NaN. Sometimes you want different behavior. The `fill_value` parameter lets you specify what to use when data is missing.

In [158]:
# Strategy returns (started trading in February)
strategy = pd.DataFrame({
    'return': [0.025, 0.018, -0.008]
}, index=pd.to_datetime(['2024-02-01', '2024-03-01', '2024-04-01']))

# Benchmark returns (full history)
benchmark = pd.DataFrame({
    'return': [0.015, 0.020, 0.012, -0.005]
}, index=pd.to_datetime(['2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01']))

print("Strategy Returns:")
print(strategy)
print("\nBenchmark Returns:")
print(benchmark)

Strategy Returns:
            return
2024-02-01   0.025
2024-03-01   0.018
2024-04-01  -0.008

Benchmark Returns:
            return
2024-01-01   0.015
2024-02-01   0.020
2024-03-01   0.012
2024-04-01  -0.005


In [159]:
# Default behavior: NaN where data is missing
excess_default = strategy - benchmark

print("Excess Returns (default):")
print(excess_default)

# Using fill_value: treat missing strategy returns as 0
excess_filled = strategy.sub(benchmark, fill_value=0)

print("\nExcess Returns (fill_value=0):")
print(excess_filled)

Excess Returns (default):
            return
2024-01-01     NaN
2024-02-01   0.005
2024-03-01   0.006
2024-04-01  -0.003

Excess Returns (fill_value=0):
            return
2024-01-01  -0.015
2024-02-01   0.005
2024-03-01   0.006
2024-04-01  -0.003


With `fill_value=0`, Pandas treats the missing January strategy return as zero. The result for January is `0 - 0.015 = -0.015`.

Use `fill_value` carefully. In this example, treating missing data as zero means the strategy underperforms the benchmark in January even though it was not trading yet. That may or may not be the comparison you want.

The default NaN behavior is usually safer because it makes missing data explicit. You can then decide how to handle it based on your specific analysis.

### 2.4 DataFrame Math Summary

You now have the tools to perform calculations across DataFrames efficiently. Here are the key takeaways:

| Operation | Syntax | When to Use |
|-----------|--------|-------------|
| Basic arithmetic | `df + 10`, `df * 2` | Applying a scalar to all values |
| Method arithmetic | `df.add()`, `df.sub()`, `df.mul()`, `df.div()` | When you need axis control |
| Row alignment | `df.sub(series, axis=0)` | Subtracting a value per date (risk-free rate) |
| Column alignment | `df.mul(series, axis=1)` | Applying weights per asset |
| Handle missing data | `df.sub(other, fill_value=0)` | When indices do not match perfectly |
| Normalization | `(df - df.mean()) / df.std()` | Scaling features for machine learning |
| Portfolio return | `df.mul(weights, axis=1).sum(axis=1)` | Weighted sum across assets |

The pattern to remember: Pandas aligns by index, then broadcasts across the remaining dimension. When alignment fails, you get NaN. When it succeeds, you get element-wise operations without loops.

This is the Pandas equivalent of NumPy broadcasting, with the added benefit of label-based alignment. Understanding both will make you dangerous with financial data.



---



## 3.0 Conclusion

You now have the Pandas foundation you need for quantitative finance. Let us recap what you learned:

**DataFrames and Series:** The core data structures for financial analysis. DataFrames hold your prices, returns, and indicators. Series hold individual columns or time series.

**Indexing and Selection:** Access data by label with `.loc[]`, by position with `.iloc[]`, and by condition with boolean indexing. These are the tools you use to slice and filter your data.

**Combining Data:** Use `pd.concat()` to stack data vertically or horizontally. Use `df.join()` and `pd.merge()` to combine datasets by index or column values. Watch for the jagged edge problem when indices do not align.

**Time Series Operations:** Rolling windows with `.rolling()`, lagging with `.shift()`, cumulative calculations with `.cumsum()` and `.cumprod()`, and frequency conversion with `.resample()`. These methods power most trading indicators.

**DataFrame Math:** Arithmetic operations with automatic index alignment and broadcasting. Subtract benchmarks, apply weights, and normalize features without writing loops.

The common thread: Pandas handles alignment and broadcasting so you can focus on the logic of your strategy rather than the mechanics of data manipulation.

### 3.1 What's Next

In Notebook 2.2, you will learn NumPy, the engine that powers Pandas. You will see why vectorization is fast, how to prepare your data for scikit-learn, and how broadcasting works at the array level.

Understanding both Pandas and NumPy gives you the complete picture of how your data flows from raw prices to trained models.